# KY CAM-ICU v11 Colab 전체 코드

**런타임 → 모두 실행**을 누르면 전체 파일을 만들고 테스트한 뒤 서버를 켭니다.

- 새 기능: LLM 기반 관찰 서술 해석(RASS 제안 + 근거 인용), 간호사 통계 권한 제한
- **AI 키:** 왼쪽 🔑 Secrets에 `GEMINI_API_KEY`(또는 `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`)를 추가하고 노트북 액세스를 켜세요. 키가 없어도 앱은 동작하고 AI만 "사용 불가"로 표시됩니다.
- DB는 `MyDrive/KY_CAM_ICU_v11/cam_icu.db`를 사용합니다.
- 교육·시연용이며 실제 임상 진단을 대신하지 않습니다.


In [ ]:
from pathlib import Path
ROOT = Path('/content/cam_icu_rass_v11')

def write(name, text):
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    print('작성:', name)


In [ ]:
write('requirements.txt', r'''Flask>=3.0.0
Werkzeug>=3.0.0
requests>=2.28.0
''')


In [ ]:
write('cam_icu_logic.py', r'''# OBSERVATION_OPTIONS: (표시 텍스트, RASS 점수) 순서로 정의
OBSERVATION_OPTIONS = [
    ("환자가 폭력적이며 의료진에게 즉각적인 위험이 있음", 4),
    ("튜브나 카테터를 당기며 의료진에게 공격적임", 3),
    ("목적 없는 움직임이 잦거나 인공호흡기에 저항함", 2),
    ("불안하거나 초조하지만 공격적이지 않음", 1),
    ("깨어 있고 차분함", 0),
    ("목소리에 눈을 뜨고 10초 이상 눈맞춤을 유지함", -1),
    ("목소리에 잠깐 눈을 뜨지만 눈맞춤이 10초 미만임", -2),
    ("목소리에 움직이거나 눈을 뜨지만 눈맞춤은 없음", -3),
    ("목소리에는 반응하지 않고 신체 자극에 움직임", -4),
    ("목소리와 신체 자극 모두에 반응이 없음", -5),
]


def derive_f3(rass_score):
    """F3(의식수준 변화)는 RASS가 0이 아니면 양성이다."""
    return rass_score != 0


def calculate_cam_icu(rass_score, f1=None, f2=None, f4=None):
    """공식: F1 AND F2 AND (F3 OR F4). F3는 RASS로 서버에서 자동 계산한다."""
    if not isinstance(rass_score, int) or not -5 <= rass_score <= 4:
        raise ValueError("RASS 점수는 -5부터 +4 사이의 정수여야 합니다.")
    if rass_score <= -4:
        return "UNASSESSABLE"
    features = (f1, f2, f4)
    if any(type(value) is not bool for value in features):
        raise ValueError("CAM-ICU 특징 F1, F2, F4를 모두 선택해야 합니다.")
    f3 = derive_f3(rass_score)
    return "POSITIVE" if f1 and f2 and (f3 or f4) else "NEGATIVE"


def suggest_rass_by_index(index):
    """표준 절차 관찰 항목을 RASS 점수로 환산한다 (규칙 매핑, AI 아님)."""
    if not isinstance(index, int) or not 0 <= index < len(OBSERVATION_OPTIONS):
        raise ValueError("유효하지 않은 관찰 항목 인덱스입니다.")
    text, score = OBSERVATION_OPTIONS[index]
    return {
        "status": "SCORED",
        "score": score,
        "reason": "선택한 관찰 항목의 RASS 정의에 따른 환산값입니다.",
    }


def suggest_rass(observation):
    """키워드 규칙 방식 (AI 비교 기준용, eval_ai.py에서 사용). 앱 화면에서는 쓰지 않는다."""
    text = (observation or "").strip().lower()
    if not text:
        return {"status": "ABSTAIN", "score": None, "reason": "관찰 기록이 없어 제안하지 않았습니다."}
    rules = [
        (("반응 없음", "무반응", "깨우지 못"), -5),
        (("신체 자극", "통증 자극"), -4),
        (("목소리", "눈을 뜸", "움직임"), -3),
        (("짧게 눈맞춤", "10초 미만"), -2),
        (("10초 이상", "지속적 눈맞춤"), -1),
        (("차분", "명료", "정상"), 0),
        (("불안", "초조"), 1),
        (("빈번한 움직임", "인공호흡기 저항"), 2),
        (("튜브를 당김", "공격적"), 3),
        (("폭력", "즉각 위험"), 4),
    ]
    for words, score in rules:
        if any(word in text for word in words):
            return {"status": "SCORED", "score": score, "reason": "관찰 기록의 표현을 참고한 시연용 제안입니다."}
    return {"status": "ABSTAIN", "score": None, "reason": "점수를 제안할 근거가 충분하지 않습니다."}
''')


In [ ]:
write('ai_rass.py', r'''"""LLM 기반 RASS 해석.

간호사가 자유롭게 쓴 관찰 서술을 RASS 공식 정의에 맞춰 해석하고,
제안 점수와 근거(서술 원문 인용)를 돌려준다. 최종 RASS는 간호사가 확정한다.

API 키 설정 (하나만 있으면 됨, 환경변수 또는 Colab Secrets):
    GEMINI_API_KEY / OPENAI_API_KEY / ANTHROPIC_API_KEY
선택 설정:
    AI_PROVIDER = gemini | openai | anthropic   (생략하면 키가 있는 쪽 사용)
    AI_MODEL    = 모델 이름                      (생략하면 아래 기본값)
"""
import json
import os
import re
import urllib.error
import urllib.request

RASS_CRITERIA = [
    (4, "공격적: 명백히 폭력적이며 의료진에게 즉각적인 위험이 있음"),
    (3, "매우 초조: 튜브나 카테터를 잡아당기거나 제거함, 공격적임"),
    (2, "초조: 목적 없는 움직임이 잦음, 인공호흡기와 맞지 않음(저항)"),
    (1, "안절부절: 불안하거나 걱정스러워하지만 움직임이 공격적이거나 격렬하지 않음"),
    (0, "깨어 있고 차분함"),
    (-1, "졸림: 완전히 깨어 있지는 않지만 목소리에 눈을 뜨고 10초 이상 눈맞춤 유지"),
    (-2, "가벼운 진정: 목소리에 잠깐 깨어나 눈맞춤하지만 10초 미만"),
    (-3, "중등도 진정: 목소리에 움직이거나 눈을 뜨지만 눈맞춤은 없음"),
    (-4, "깊은 진정: 목소리에는 반응 없고 신체 자극에 움직이거나 눈을 뜸"),
    (-5, "깨울 수 없음: 목소리와 신체 자극 모두에 반응 없음"),
]

DEFAULT_MODELS = {
    "gemini": "gemini-2.5-flash",
    "openai": "gpt-4.1-mini",
    "anthropic": "claude-haiku-4-5",
}
KEY_NAMES = {
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
MAX_NARRATIVE = 600
TIMEOUT_SECONDS = 25


def build_prompt(narrative):
    criteria = "\n".join(f"{score:+d}: {text}" for score, text in RASS_CRITERIA)
    return f"""너는 중환자실 간호사의 RASS(Richmond Agitation-Sedation Scale) 평가를 돕는 보조자다.
아래 [관찰 서술]만 근거로, [RASS 기준] 중 어디에 해당하는지 판단하라.

규칙:
1. 서술에 적힌 사실만 사용한다. 적히지 않은 반응, 자극, 시간은 추측하지 않는다.
2. 점수를 정하려면 자극 단계(무자극/목소리/신체 자극)와 반응이 서술에 있어야 한다.
   깨어 있지 않은 환자에서 눈맞춤 시간이 10초 이상인지 미만인지 알 수 없으면 -1과 -2를 구분하지 않는다.
3. 근거가 부족하면 score를 null로 하고 missing에 부족한 정보를 적는다.
4. evidence에는 판단 근거가 된 [관찰 서술]의 구절을 **한 글자도 바꾸지 말고** 그대로 복사한다.
5. 섬망 여부나 치료는 판단하지 않는다.

[RASS 기준]
{criteria}

[관찰 서술]
{narrative}

다음 JSON 하나만 출력하라. 다른 글은 쓰지 마라.
{{"score": 정수(-5~4) 또는 null, "criterion": "해당 기준 문장", "evidence": "서술 원문 인용", "missing": "부족한 정보 또는 빈 문자열"}}"""


def detect_provider():
    provider = os.environ.get("AI_PROVIDER", "").strip().lower()
    if provider in KEY_NAMES:
        return provider if os.environ.get(KEY_NAMES[provider]) else None
    for name, key in KEY_NAMES.items():
        if os.environ.get(key):
            return name
    return None


def _post(url, payload, headers):
    request = urllib.request.Request(
        url, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", **headers}, method="POST")
    with urllib.request.urlopen(request, timeout=TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode("utf-8"))


def call_llm(provider, prompt):
    """선택한 서비스에 프롬프트를 보내고 응답 텍스트를 돌려준다."""
    key = os.environ[KEY_NAMES[provider]]
    model = os.environ.get("AI_MODEL") or DEFAULT_MODELS[provider]
    if provider == "gemini":
        data = _post(
            f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
            {"contents": [{"parts": [{"text": prompt}]}],
             "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}},
            {"x-goog-api-key": key})
        return data["candidates"][0]["content"]["parts"][0]["text"]
    if provider == "openai":
        data = _post(
            "https://api.openai.com/v1/chat/completions",
            {"model": model, "messages": [{"role": "user", "content": prompt}],
             "response_format": {"type": "json_object"}},
            {"Authorization": f"Bearer {key}"})
        return data["choices"][0]["message"]["content"]
    data = _post(
        "https://api.anthropic.com/v1/messages",
        {"model": model, "max_tokens": 400, "temperature": 0,
         "messages": [{"role": "user", "content": prompt}]},
        {"x-api-key": key, "anthropic-version": "2023-06-01"})
    return data["content"][0]["text"]


def _normalize(text):
    return re.sub(r"[\s.,·!?~\"'“”‘’]+", "", text or "")


def parse_response(raw, narrative):
    """LLM 응답을 검증한다. 형식이 틀리거나 근거 인용이 원문에 없으면 점수를 버린다."""
    match = re.search(r"\{.*\}", raw or "", re.S)
    if not match:
        return _result("ERROR", None, "AI 응답 형식을 해석하지 못했습니다.")
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return _result("ERROR", None, "AI 응답 형식을 해석하지 못했습니다.")

    score = data.get("score")
    criterion = str(data.get("criterion") or "").strip()
    evidence = str(data.get("evidence") or "").strip()
    missing = str(data.get("missing") or "").strip()

    if score is None:
        reason = f"판단 보류: {missing}" if missing else "판단 보류: 서술에 근거가 부족합니다."
        return _result("ABSTAIN", None, reason)
    if type(score) is not int or not -5 <= score <= 4:
        return _result("ERROR", None, "AI가 RASS 범위를 벗어난 값을 냈습니다.")
    if not evidence or _normalize(evidence) not in _normalize(narrative):
        return _result("ABSTAIN", None,
                       "판단 보류: AI가 제시한 근거가 관찰 서술 원문에 없습니다(환각 차단).")
    return _result("SCORED", score, f"기준 {score:+d} — {criterion}", evidence)


def _result(status, score, reason, evidence=""):
    return {"status": status, "score": score, "reason": reason, "evidence": evidence}


def suggest_rass_ai(narrative, caller=call_llm):
    narrative = (narrative or "").strip()
    if not narrative:
        return _result("ABSTAIN", None, "관찰 서술을 입력해야 AI가 해석할 수 있습니다.")
    if len(narrative) > MAX_NARRATIVE:
        return _result("ABSTAIN", None, f"관찰 서술은 {MAX_NARRATIVE}자 이내로 입력해 주세요.")
    provider = detect_provider()
    if provider is None:
        return _result("UNAVAILABLE", None,
                       "AI API 키가 설정되지 않았습니다. README의 AI 설정을 확인하세요.")
    try:
        raw = caller(provider, build_prompt(narrative))
    except urllib.error.HTTPError as e:
        return _result("ERROR", None, f"AI 서비스 오류 (HTTP {e.code}). 키와 모델 이름을 확인하세요.")
    except Exception as e:
        return _result("ERROR", None, f"AI 서비스에 연결하지 못했습니다 ({type(e).__name__}).")
    return parse_response(raw, narrative)
''')


In [ ]:
write('database.py', r'''"""
database.py — KY CAM-ICU/RASS v5

가상 환자 데이터 생성 기준 (근거 문헌):
─────────────────────────────────────────────────────────────────
[1] Ely EW et al. JAMA 2001;286(21):2703-2710
    · CAM-ICU 원논문. 기계호흡 환자 83.3% 섬망 발생.

[2] Sessler CN et al. Am J Respir Crit Care Med. 2002;166:1338-1344
    · RASS 원논문. 중증도별 점수 분포 기준.

[3] Krewulak KD et al. Crit Care Med. 2018;46(12):2028-2035
    · CAM-ICU 메타분석(n=16,595): 섬망 유병률 31%.

[4] Khan SH et al. Heart Lung. 2020;49(4):572-577
    · ICU 섬망률 16.5%, 혼수(RASS≤-4) 17-24%.

[5] Delirium Journal. 2023; n=20,241명:
    · CAM-ICU POSITIVE 25.1%.

[6] Liu Y et al. Crit Care Med. 2019 (n=23,062):
    · ICU 평균 연령 52-67세, 남성 57.5%.

[7] Salluh JIF et al. BMC Crit Care. 2010:
    · ICU 진단 분포 — 패혈증 47%, 폐렴 14%, 외상 등.

생성 파라미터 요약:
  CAM-ICU 결과: POSITIVE 28%, NEGATIVE 57%, UNASSESSABLE 15%
  평균 연령: 62세 (18-44: 18%, 45-64: 42%, 65-79: 30%, 80+: 10%)
  성별: 남성 60%, 여성 40%  [Liu 2019]
  진정제 사용: 60%
  RASS 분포: 0(-1-0 중심, 중앙값 ~-1)  [Sessler 2002]
─────────────────────────────────────────────────────────────────
"""

import os
import random
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path
from werkzeug.security import generate_password_hash
from cam_icu_logic import calculate_cam_icu

BASE_DIR = Path(__file__).resolve().parent
DB_PATH  = Path(os.environ.get("CAM_ICU_DB_PATH", BASE_DIR / "data" / "cam_icu.db"))

# ── 이름 생성 데이터 ───────────────────────────────────
_SURNAMES = ['김','이','박','최','정','강','조','윤','임','한','오','신','권','황','안',
             '서','류','남','홍','배','문','노','석','양','진','구','변','성','장','표',
             '심','차','방','추','곽','함','엄','도','나','송','천','고','손','하','허',
             '염','주','현','기','전','민','유','백','라','어','마','제','탁','봉','나']
_MALE    = ['민준','지훈','해준','현우','도현','승민','준혁','지호','재원','태현',
            '현준','도윤','성우','민혁','재훈','시우','정우','건우','주원','영재',
            '민수','정호','진우','태준','민호','준서','현서','도준','수호','재윤',
            '시준','지환','성진','우진','태양','기태','선우','이준','지안','건호']
_FEMALE  = ['서연','유진','소희','지원','지수','혜린','미영','소은','수빈','채원',
            '지영','미래','지안','유나','지혜','은지','혜수','나은','예린','수아',
            '지민','소연','하은','민지','수정','예지','다은','지현','소정','예은',
            '아름','나리','보미','수현','혜인','나현','소현','예나','채은','하린']

# ── 진단명 분포 (Salluh 2010; Ely 2001 기반) ─────────
# 패혈증 25%, 폐렴 20%, 수술후 15%, 급성호흡부전 10%,
# 심부전 8%, 뇌졸중 7%, 외상 6%, 신부전 3%, 뇌출혈 3%, 기타 3%
_DIAG_POOL = (
    ['패혈증'] * 25 +
    ['폐렴'] * 20 +
    ['수술 후 관리'] * 15 +
    ['급성 호흡부전'] * 10 +
    ['심부전'] * 8 +
    ['뇌졸중'] * 7 +
    ['외상'] * 6 +
    ['신부전'] * 3 +
    ['뇌출혈'] * 3 +
    ['기타 내과적 질환'] * 3
)

# ── RASS 분포 가중치 (Sessler 2002; 중앙값 -1) ───────
# +4:1%, +3:2%, +2:4%, +1:8%, 0:25%, -1:22%, -2:16%, -3:10%, -4:7%, -5:5%
_RASS_CHOICES = (
    [4]*1 + [3]*2 + [2]*4 + [1]*8 +
    [0]*25 + [-1]*22 + [-2]*16 + [-3]*10 +
    [-4]*7 + [-5]*5
)

# ── 관찰 항목 텍스트 ──────────────────────────────────
_OBS = [
    '환자가 폭력적이며 의료진에게 즉각적인 위험이 있음',     # +4
    '튜브나 카테터를 당기며 의료진에게 공격적임',             # +3
    '목적 없는 움직임이 잦거나 인공호흡기에 저항함',          # +2
    '불안하거나 초조하지만 공격적이지 않음',                  # +1
    '깨어 있고 차분함',                                       # 0
    '목소리에 눈을 뜨고 10초 이상 눈맞춤을 유지함',           # -1
    '목소리에 잠깐 눈을 뜨지만 눈맞춤이 10초 미만임',         # -2
    '목소리에 움직이거나 눈을 뜨지만 눈맞춤은 없음',          # -3
    '목소리에는 반응하지 않고 신체 자극에 움직임',             # -4
    '목소리와 신체 자극 모두에 반응이 없음',                  # -5
]
_AI_RASS = [4, 3, 2, 1, 0, -1, -2, -3, -4, -5]

# ── 기존 74명 실제 환자 ───────────────────────────────
_PATIENTS_A = [
    ("P-A001","김민준","ICU-A01",25,"남",1), ("P-A002","이서연","ICU-A02",32,"여",1),
    ("P-A003","박지훈","ICU-A03",45,"남",1), ("P-A004","최유진","ICU-A04",28,"여",1),
    ("P-A005","정해준","ICU-A05",67,"남",1), ("P-A006","강소희","ICU-A06",52,"여",1),
    ("P-A007","조현우","ICU-A07",38,"남",1), ("P-A008","윤지원","ICU-A08",71,"여",1),
    ("P-A009","임도현","ICU-A09",23,"남",1), ("P-A010","한지수","ICU-A10",56,"여",1),
    ("P-A011","오승민","ICU-A11",44,"남",1), ("P-A012","신혜린","ICU-A12",35,"여",1),
    ("P-A013","권태양","ICU-A13",62,"남",1), ("P-A014","황미영","ICU-A14",48,"여",1),
    ("P-A015","안준혁","ICU-A15",31,"남",1), ("P-A016","서보라","ICU-A16",79,"여",1),
    ("P-A017","류지호","ICU-A17",22,"남",1), ("P-A018","남소은","ICU-A18",58,"여",1),
    ("P-A019","문재원","ICU-A19",73,"남",1), ("P-A020","배채원","ICU-A20",36,"여",1),
    ("P-A021","노태현","ICU-A21",55,"남",1), ("P-A022","석지영","ICU-A22",29,"여",1),
    ("P-A023","양현준","ICU-A23",66,"남",1), ("P-A024","진미래","ICU-A24",43,"여",1),
    ("P-A025","구도윤","ICU-A25",21,"남",1), ("P-A026","홍지안","ICU-A26",77,"여",1),
    ("P-A027","변성우","ICU-A27",50,"남",1), ("P-A028","성유나","ICU-A28",33,"여",1),
    ("P-A029","장민혁","ICU-A29",60,"남",1), ("P-A030","표지혜","ICU-A30",47,"여",1),
    ("P-A031","심기태","ICU-A31",27,"남",1), ("P-A032","차은지","ICU-A32",69,"여",1),
    ("P-A033","방재훈","ICU-A33",39,"남",1), ("P-A034","추수아","ICU-A34",54,"여",1),
    ("P-A035","곽시우","ICU-A35",76,"남",1), ("P-A036","함지민","ICU-A36",41,"여",1),
    ("P-A037","엄정우","ICU-A37",26,"남",1), ("P-A038","도소연","ICU-A38",63,"여",1),
    ("P-A039","나건우","ICU-A39",80,"남",1), ("P-A040","송나은","ICU-A40",37,"여",1),
]
_PATIENTS_B = [
    ("P-B001","천준호","ICU-B01",42,"남",2), ("P-B002","고하은","ICU-B02",57,"여",2),
    ("P-B003","손예린","ICU-B03",29,"여",2), ("P-B004","하지훈","ICU-B04",65,"남",2),
    ("P-B005","허유나","ICU-B05",34,"여",2), ("P-B006","염민수","ICU-B06",51,"남",2),
    ("P-B007","주미래","ICU-B07",78,"여",2), ("P-B008","현태준","ICU-B08",24,"남",2),
    ("P-B009","기소영","ICU-B09",43,"여",2), ("P-B010","최재혁","ICU-B10",70,"남",2),
    ("P-B011","전수연","ICU-B11",38,"여",2), ("P-B012","박민준","ICU-B12",55,"남",2),
    ("P-B013","이지수","ICU-B13",27,"여",2), ("P-B014","정영호","ICU-B14",63,"남",2),
    ("P-B015","강은아","ICU-B15",46,"여",2), ("P-B016","조성현","ICU-B16",32,"남",2),
    ("P-B017","윤수진","ICU-B17",75,"여",2), ("P-B018","임태호","ICU-B18",48,"남",2),
    ("P-B019","한지은","ICU-B19",22,"여",2), ("P-B020","오민호","ICU-B20",59,"남",2),
    ("P-B021","신예슬","ICU-B21",36,"여",2), ("P-B022","권재민","ICU-B22",68,"남",2),
    ("P-B023","황혜진","ICU-B23",41,"여",2), ("P-B024","안승준","ICU-B24",26,"남",2),
    ("P-B025","서다영","ICU-B25",72,"여",2), ("P-B026","류우혁","ICU-B26",53,"남",2),
    ("P-B027","남소라","ICU-B27",31,"여",2), ("P-B028","홍준영","ICU-B28",44,"남",2),
    ("P-B029","배지현","ICU-B29",67,"여",2), ("P-B030","문민재","ICU-B30",28,"남",2),
    ("P-B031","노하영","ICU-B31",80,"여",2), ("P-B032","석정호","ICU-B32",37,"남",2),
    ("P-B033","양수민","ICU-B33",61,"여",2), ("P-B034","진성진","ICU-B34",49,"남",2),
]


@contextmanager
def connect_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def init_db():
    with connect_db() as db:
        db.execute("""CREATE TABLE IF NOT EXISTS wards(
            id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL UNIQUE)""")
        db.execute("""CREATE TABLE IF NOT EXISTS users(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE, password_hash TEXT NOT NULL,
            name TEXT NOT NULL, employee_no TEXT NOT NULL UNIQUE,
            ward_id INTEGER REFERENCES wards(id),
            role TEXT NOT NULL DEFAULT 'NURSE',
            last_login TEXT,
            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)""")
        db.execute("""CREATE TABLE IF NOT EXISTS patients(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_code TEXT NOT NULL UNIQUE, name TEXT NOT NULL,
            room_number TEXT NOT NULL, age INTEGER NOT NULL,
            sex TEXT NOT NULL CHECK(sex IN ('남','여')),
            ward_id INTEGER NOT NULL REFERENCES wards(id),
            status TEXT NOT NULL DEFAULT '입원중',
            diagnosis TEXT DEFAULT '',
            admission_date TEXT DEFAULT '',
            sedative_use INTEGER DEFAULT 0)""")
        db.execute("""CREATE TABLE IF NOT EXISTS patient_assignments(
            user_id INTEGER NOT NULL REFERENCES users(id),
            patient_id INTEGER NOT NULL REFERENCES patients(id),
            PRIMARY KEY(user_id, patient_id))""")
        db.execute("""CREATE TABLE IF NOT EXISTS assessments(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER NOT NULL REFERENCES patients(id),
            nurse_id INTEGER NOT NULL REFERENCES users(id),
            assessed_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
            observation TEXT NOT NULL,
            narrative TEXT DEFAULT '',
            ai_status TEXT NOT NULL, ai_rass INTEGER,
            ai_reason TEXT DEFAULT '', ai_evidence TEXT DEFAULT '',
            rass_score INTEGER NOT NULL CHECK(rass_score BETWEEN -5 AND 4),
            feature1 INTEGER, feature2 INTEGER, feature3 INTEGER, feature4 INTEGER,
            cam_result TEXT NOT NULL CHECK(cam_result IN ('POSITIVE','NEGATIVE','UNASSESSABLE')),
            memo TEXT DEFAULT '')""")
        # v10 이하 DB에도 새 칼럼 추가
        cols = {r[1] for r in db.execute("PRAGMA table_info(assessments)")}
        for col in ("narrative", "ai_reason", "ai_evidence"):
            if col not in cols:
                db.execute(f"ALTER TABLE assessments ADD COLUMN {col} TEXT DEFAULT ''")
        db.execute("""CREATE TABLE IF NOT EXISTS audit_logs(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER, action TEXT NOT NULL, detail TEXT,
            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)""")

        # 마이그레이션
        for tbl, col, defn in [
            ("patients",    "status",         "TEXT NOT NULL DEFAULT '입원중'"),
            ("patients",    "diagnosis",      "TEXT DEFAULT ''"),
            ("patients",    "admission_date", "TEXT DEFAULT ''"),
            ("patients",    "sedative_use",   "INTEGER DEFAULT 0"),
            ("assessments", "memo",           "TEXT DEFAULT ''"),
            ("users",       "last_login",     "TEXT"),
        ]:
            try:
                db.execute(f"ALTER TABLE {tbl} ADD COLUMN {col} {defn}")
            except Exception:
                pass

        # 기본 시드
        if db.execute("SELECT COUNT(*) FROM wards").fetchone()[0] == 0:
            db.executemany("INSERT INTO wards(name) VALUES(?)",
                           [("중환자실 A",), ("중환자실 B",)])

        if db.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0:
            pw    = generate_password_hash("1234",      method="pbkdf2:sha256")
            admpw = generate_password_hash("admin1234", method="pbkdf2:sha256")
            db.executemany(
                "INSERT INTO users(username,password_hash,name,employee_no,ward_id,role)"
                " VALUES(?,?,?,?,?,?)",
                [("admin",   admpw, "관리자", "A-0001", None, "ADMIN"),
                 ("nurse01", pw,    "김간호", "N-1001", 1,    "NURSE"),
                 ("nurse02", pw,    "이간호", "N-1002", 2,    "NURSE")])

        if db.execute("SELECT COUNT(*) FROM patients").fetchone()[0] == 0:
            db.executemany(
                "INSERT INTO patients(patient_code,name,room_number,age,sex,ward_id)"
                " VALUES(?,?,?,?,?,?)",
                _PATIENTS_A + _PATIENTS_B)
            a_ids = [r[0] for r in db.execute(
                "SELECT id FROM patients WHERE ward_id=1 ORDER BY patient_code").fetchall()]
            b_ids = [r[0] for r in db.execute(
                "SELECT id FROM patients WHERE ward_id=2 ORDER BY patient_code").fetchall()]
            n1 = db.execute("SELECT id FROM users WHERE username='nurse01'").fetchone()[0]
            n2 = db.execute("SELECT id FROM users WHERE username='nurse02'").fetchone()[0]
            db.executemany("INSERT INTO patient_assignments(user_id,patient_id) VALUES(?,?)",
                           [(n1, p) for p in a_ids] + [(n2, p) for p in b_ids])

        # 가상 환자 1만명 (최초 1회)
        if db.execute("SELECT COUNT(*) FROM patients").fetchone()[0] < 1000:
            print("[DB] 문헌 기반 가상 환자 1만명 생성 중...", flush=True)
            _seed_bulk_patients(db)
            print("[DB] 문헌 기반 평가 기록 생성 중...", flush=True)
            _seed_bulk_assessments(db)
            print("[DB] 완료 ✅  (근거: Ely 2001, Sessler 2002, Krewulak 2018)", flush=True)


def _gen_age(rng):
    """
    연령 분포 (Liu et al., 2019; ICU 연구 종합):
    18-44세 18%, 45-64세 42%, 65-79세 30%, 80+세 10%
    → 평균 연령 약 62세 (문헌: 57~67세)
    """
    # 연령 분포 (Liu 2019: 45-64y 37%, 65+y 27%; 한국 ICU 고령화 반영)
    # 18-44:12%, 45-64:40%, 65-79:36%, 80+:12% → 평균 약 62세
    groups = [
        (20, 44, 12),
        (45, 64, 40),
        (65, 79, 36),
        (80, 85, 12),
    ]
    r = rng.randint(1, 100)
    cumsum = 0
    for lo, hi, w in groups:
        cumsum += w
        if r <= cumsum:
            return rng.randint(lo, hi)
    return rng.randint(80, 85)


def _seed_bulk_patients(db):
    """
    성별: 남 60% / 여 40%  (Liu 2019: 57.5% male)
    진정제: 60% 사용  (ICU 일반적 진정 요법 비율)
    상태: 입원중 70% / 퇴원 25% / 외출 5%
    진단: Salluh 2010, Ely 2001 진단 분포 반영
    """
    rng = random.Random(2025)
    existing = db.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
    need = 10000 - existing
    if need <= 0:
        return
    rows = []
    for i in range(need):
        # 성별: 남 60%, 여 40%  [Liu 2019]
        sex  = '남' if rng.random() < 0.60 else '여'
        name = rng.choice(_SURNAMES) + rng.choice(_MALE if sex == '남' else _FEMALE)
        age  = _gen_age(rng)
        ward_id = 1 if i < need // 2 else 2
        letter  = 'A' if ward_id == 1 else 'B'
        room    = f"ICU-{letter}{(i % 200) + 1:03d}"
        code    = f"P-{existing + i + 1:05d}"
        status  = rng.choices(['입원중','퇴원','외출'], weights=[70,25,5])[0]
        diag    = rng.choice(_DIAG_POOL)
        base    = datetime(2024, 1, 1)
        adm     = (base + timedelta(days=rng.randint(0, 400))).strftime('%Y-%m-%d')
        sed     = 1 if rng.random() < 0.60 else 0  # 60% 진정제 사용
        rows.append((code, name, room, age, sex, ward_id, status, diag, adm, sed))
    db.executemany(
        "INSERT OR IGNORE INTO patients"
        "(patient_code,name,room_number,age,sex,ward_id,status,diagnosis,admission_date,sedative_use)"
        " VALUES(?,?,?,?,?,?,?,?,?,?)", rows)


def _seed_bulk_assessments(db):
    """
    CAM-ICU 결과 분포 (문헌 근거):
      POSITIVE    28%  (Krewulak 2018 메타분석 31%; 대규모 연구 n=20,241에서 25.1%)
      NEGATIVE    57%  (평가 가능 환자 중 비섬망)
      UNASSESSABLE 15% (RASS≤-4; Khan 2020: coma 17-24%)

    RASS 분포 (Sessler 2002 기반):
      +4:1%, +3:2%, +2:4%, +1:8%, 0:25%, -1:22%,
      -2:16%, -3:10%, -4:7%, -5:5%  → 중앙값 약 -1

    평가 빈도: 환자 40% 대상, 1-4회 평가 (8시간마다 1회 기준)
    """
    rng = random.Random(2025)
    nurse_ids = [r[0] for r in
                 db.execute("SELECT id FROM users WHERE role='NURSE'").fetchall()]
    if not nurse_ids:
        return

    all_pids = [r[0] for r in db.execute("SELECT id FROM patients").fetchall()]
    sample   = rng.sample(all_pids, min(4000, len(all_pids)))
    now      = datetime.now()
    base     = now - timedelta(days=180)

    rows = []
    for pid in sample:
        for _ in range(rng.randint(1, 4)):
            nid     = rng.choice(nurse_ids)
            rass    = rng.choice(_RASS_CHOICES)
            obs_idx = _rass_to_obs_idx(rass)
            obs     = _OBS[obs_idx]
            dt      = base + timedelta(
                seconds=rng.randint(0, int((now - base).total_seconds())))
            assessed_at = dt.strftime('%Y-%m-%d %H:%M:%S')

            if rass <= -4:
                # UNASSESSABLE (15% — RASS -4,-5)
                cam = 'UNASSESSABLE'
                f1 = f2 = f3 = f4 = None
            else:
                f3 = 0 if rass == 0 else 1          # RASS가 0이 아니면 F3 양성
                if rng.random() < 0.33:
                    f1, f2 = 1, 1
                    f4 = 1 if f3 == 0 else rng.choice([0, 1])
                else:
                    patterns = ['no_f1', 'no_f2'] if f3 else ['no_f1', 'no_f2', 'no_f4']
                    pattern = rng.choice(patterns)
                    if pattern == 'no_f1':
                        f1, f2, f4 = 0, rng.choice([0, 1]), rng.choice([0, 1])
                    elif pattern == 'no_f2':
                        f1, f2, f4 = rng.choice([0, 1]), 0, rng.choice([0, 1])
                    else:
                        f1, f2, f4 = rng.choice([0, 1]), rng.choice([0, 1]), 0
                cam = calculate_cam_icu(rass, bool(f1), bool(f2), bool(f4))

            rows.append((pid, nid, assessed_at, obs, 'NOT_RUN', None,
                         rass, f1, f2, f3, f4, cam, ''))

    db.executemany(
        "INSERT INTO assessments"
        "(patient_id,nurse_id,assessed_at,observation,ai_status,ai_rass,"
        "rass_score,feature1,feature2,feature3,feature4,cam_result,memo)"
        " VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)", rows)


def _rass_to_obs_idx(rass):
    """RASS 점수 → 관찰 항목 인덱스 (Sessler 2002 RASS 정의 기반)."""
    return {4:0, 3:1, 2:2, 1:3, 0:4, -1:5, -2:6, -3:7, -4:8, -5:9}[rass]


def log_action(user_id, action, detail=""):
    with connect_db() as db:
        db.execute("INSERT INTO audit_logs(user_id,action,detail) VALUES(?,?,?)",
                   (user_id, action, detail))
''')


In [ ]:
write('app.py', r'''import json, os, sys, traceback
from datetime import datetime, timedelta, timezone
from functools import wraps
from flask import Flask, abort, flash, redirect, render_template, request, send_from_directory, session, url_for
from werkzeug.security import check_password_hash, generate_password_hash
from cam_icu_logic import calculate_cam_icu, derive_f3, OBSERVATION_OPTIONS
from ai_rass import suggest_rass_ai
from database import connect_db, init_db, log_action

app = Flask(__name__)
app.config["SECRET_KEY"] = os.environ.get("SECRET_KEY", "cam-icu-rass-2025")

print("[startup] DB 초기화 중...", flush=True)
init_db()
print("[startup] 완료", flush=True)

SORT_OPTIONS = {
    "room":     ("병실 순",        "room_number ASC"),
    "risk":     ("위험 순",        "CASE last_res WHEN 'POSITIVE' THEN 0 WHEN 'NEGATIVE' THEN 1 WHEN 'UNASSESSABLE' THEN 2 ELSE 3 END"),
    "reassess": ("재평가 필요 순", "(last_at IS NOT NULL), last_at ASC"),
    "name":     ("이름 순",        "name ASC"),
    "recent":   ("최근 검사 순",   "(last_at IS NULL), last_at DESC"),
}

# ── 데코레이터 ─────────────────────────────────────────
def login_required(f):
    @wraps(f)
    def w(*a, **k):
        if "user_id" not in session:
            return redirect(url_for("login"))
        return f(*a, **k)
    return w

def nurse_required(f):
    @wraps(f)
    def w(*a, **k):
        if "user_id" not in session:
            return redirect(url_for("login"))
        u = get_current_user()
        if not u or u["role"] != "NURSE":
            abort(403)
        return f(*a, **k)
    return w

def admin_required(f):
    @wraps(f)
    def w(*a, **k):
        if "user_id" not in session:
            return redirect(url_for("login"))
        u = get_current_user()
        if not u or u["role"] != "ADMIN":
            abort(403)
        return f(*a, **k)
    return w

# ── 헬퍼 ──────────────────────────────────────────────
def get_current_user():
    if "user_id" not in session:
        return None
    with connect_db() as db:
        return db.execute(
            "SELECT u.*, COALESCE(w.name,'관리자') AS ward_name "
            "FROM users u LEFT JOIN wards w ON w.id=u.ward_id WHERE u.id=?",
            (session["user_id"],)).fetchone()

def get_assigned_patient(patient_id):
    with connect_db() as db:
        return db.execute(
            "SELECT p.*, w.name AS ward_name FROM patients p "
            "JOIN wards w ON w.id=p.ward_id "
            "JOIN patient_assignments pa ON pa.patient_id=p.id "
            "WHERE p.id=? AND pa.user_id=?",
            (patient_id, session["user_id"])).fetchone()

def get_patients_sorted(user_id, sort):
    _, sql = SORT_OPTIONS.get(sort, SORT_OPTIONS["room"])
    with connect_db() as db:
        return db.execute(
            "WITH pd AS ("
            "  SELECT p.*,"
            "    (SELECT cam_result FROM assessments a WHERE a.patient_id=p.id ORDER BY a.id DESC LIMIT 1) AS last_res,"
            "    (SELECT assessed_at FROM assessments a WHERE a.patient_id=p.id ORDER BY a.id DESC LIMIT 1) AS last_at"
            "  FROM patients p"
            "  JOIN patient_assignments pa ON pa.patient_id=p.id"
            "  WHERE pa.user_id=?"
            f") SELECT * FROM pd ORDER BY {sql}",
            (user_id,)).fetchall()

@app.context_processor
def inject_user():
    return {"current_user": get_current_user()}

# ── 인증 ──────────────────────────────────────────────
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form.get("username", "")
        password = request.form.get("password", "")
        with connect_db() as db:
            user = db.execute("SELECT * FROM users WHERE username=?", (username,)).fetchone()
        if user and check_password_hash(user["password_hash"], password):
            session.clear()
            session["user_id"] = user["id"]
            with connect_db() as db:
                db.execute("UPDATE users SET last_login=datetime('now') WHERE id=?", (user["id"],))
            log_action(user["id"], "LOGIN")
            return redirect(url_for("admin") if user["role"] == "ADMIN" else url_for("dashboard"))
        flash("아이디 또는 비밀번호를 확인해 주세요.", "error")
    return render_template("login.html")

@app.route("/logout")
def logout():
    uid = session.get("user_id")
    session.clear()
    if uid:
        log_action(uid, "LOGOUT")
    return redirect(url_for("login"))

# ── 관리자: 간호사 관리 ───────────────────────────────
@app.route("/admin")
@admin_required
def admin():
    with connect_db() as db:
        nurses = db.execute(
            "SELECT u.*, COALESCE(w.name,'—') AS ward_name,"
            " (SELECT COUNT(*) FROM patient_assignments pa WHERE pa.user_id=u.id) AS patient_count"
            " FROM users u LEFT JOIN wards w ON w.id=u.ward_id"
            " WHERE u.role='NURSE' ORDER BY u.id").fetchall()
        wards = db.execute("SELECT * FROM wards ORDER BY id").fetchall()
    return render_template("admin.html", nurses=nurses, wards=wards)

@app.route("/admin/nurses/new", methods=["GET", "POST"])
@admin_required
def register_nurse():
    with connect_db() as db:
        wards = db.execute("SELECT * FROM wards ORDER BY id").fetchall()
    if request.method == "POST":
        username    = request.form.get("username", "").strip()
        name        = request.form.get("name", "").strip()
        employee_no = request.form.get("employee_no", "").strip()
        ward_id     = request.form.get("ward_id", "")
        password    = request.form.get("password", "")
        confirm     = request.form.get("confirm", "")
        if not all([username, name, employee_no, ward_id, password]):
            flash("모든 항목을 입력해 주세요.", "error")
        elif password != confirm:
            flash("비밀번호가 일치하지 않습니다.", "error")
        elif len(password) < 4:
            flash("비밀번호는 4자 이상이어야 합니다.", "error")
        else:
            try:
                pw = generate_password_hash(password, method="pbkdf2:sha256")
                with connect_db() as db:
                    db.execute(
                        "INSERT INTO users(username,password_hash,name,employee_no,ward_id,role)"
                        " VALUES(?,?,?,?,?,'NURSE')",
                        (username, pw, name, employee_no, int(ward_id)))
                log_action(session["user_id"], "NURSE_REGISTERED", f"username={username}")
                flash(f"'{name}' 간호사 계정이 생성되었습니다.", "success")
                return redirect(url_for("admin"))
            except Exception as e:
                flash("이미 사용 중인 아이디 또는 사번입니다." if "UNIQUE" in str(e) else f"오류: {e}", "error")
    return render_template("nurse_register.html", wards=wards)

@app.route("/admin/nurses/<int:nurse_id>/delete", methods=["POST"])
@admin_required
def delete_nurse(nurse_id):
    with connect_db() as db:
        nurse = db.execute(
            "SELECT * FROM users WHERE id=? AND role='NURSE'", (nurse_id,)).fetchone()
    if not nurse:
        flash("존재하지 않는 계정입니다.", "error")
        return redirect(url_for("admin"))
    if request.form.get("confirm_username", "") != nurse["username"]:
        flash("아이디가 일치하지 않습니다.", "error")
        return redirect(url_for("admin"))
    if not check_password_hash(nurse["password_hash"], request.form.get("confirm_password", "")):
        flash("비밀번호가 일치하지 않습니다.", "error")
        return redirect(url_for("admin"))
    with connect_db() as db:
        db.execute("DELETE FROM patient_assignments WHERE user_id=?", (nurse_id,))
        db.execute("DELETE FROM users WHERE id=?", (nurse_id,))
    log_action(session["user_id"], "NURSE_DELETED", f"username={nurse['username']}")
    flash(f"'{nurse['name']}' 간호사 계정을 삭제했습니다.", "success")
    return redirect(url_for("admin"))

# ── 관리자: 환자 관리 ─────────────────────────────────
@app.route("/admin/patients")
@admin_required
def admin_patients():
    q           = request.args.get("q", "").strip()
    ward_filter = request.args.get("ward", "").strip()
    page        = max(1, int(request.args.get("page", 1)))
    per         = 50
    off         = (page - 1) * per

    with connect_db() as db:
        wards  = db.execute("SELECT * FROM wards ORDER BY id").fetchall()
        nurses = db.execute("SELECT * FROM users WHERE role='NURSE' ORDER BY name").fetchall()

        # 병동별 환자 수 (탭 배지용)
        ward_counts = {r[0]: r[1] for r in db.execute(
            "SELECT ward_id, COUNT(*) FROM patients GROUP BY ward_id").fetchall()}
        total_count = db.execute("SELECT COUNT(*) FROM patients").fetchone()[0]

        base_q = (
            "SELECT p.*, w.name AS ward_name,"
            " (SELECT u.name FROM patient_assignments pa JOIN users u ON u.id=pa.user_id"
            "  WHERE pa.patient_id=p.id LIMIT 1) AS nurse_name"
            " FROM patients p JOIN wards w ON w.id=p.ward_id"
        )

        conditions = []
        params     = []

        if q:
            like = f"%{q}%"
            conditions.append("(p.name LIKE ? OR p.patient_code LIKE ? OR p.room_number LIKE ?)")
            params += [like, like, like]

        if ward_filter:
            conditions.append("p.ward_id=?")
            params.append(int(ward_filter))

        where = ("WHERE " + " AND ".join(conditions)) if conditions else ""

        count_q = f"SELECT COUNT(*) FROM patients p {where}"
        total   = db.execute(count_q, params).fetchone()[0]

        patients = db.execute(
            f"{base_q} {where} ORDER BY p.patient_code LIMIT ? OFFSET ?",
            params + [per, off]).fetchall()

    return render_template("admin_patients.html",
                           patients=patients, q=q, page=page,
                           ward_filter=ward_filter,
                           total=total, total_pages=(total + per - 1) // per,
                           wards=wards, nurses=nurses,
                           ward_counts=ward_counts, total_count=total_count)

@app.route("/admin/patients/new", methods=["GET", "POST"])
@admin_required
def admin_patient_new():
    with connect_db() as db:
        wards  = db.execute("SELECT * FROM wards ORDER BY id").fetchall()
        nurses = db.execute("SELECT * FROM users WHERE role='NURSE' ORDER BY name").fetchall()
        max_n  = db.execute(
            "SELECT MAX(CAST(REPLACE(patient_code,'P-','') AS INTEGER))"
            " FROM patients WHERE patient_code LIKE 'P-%'").fetchone()[0] or 0
    default_code = f"P-{max_n + 1:05d}"
    if request.method == "POST":
        f = request.form
        try:
            with connect_db() as db:
                db.execute(
                    "INSERT INTO patients(patient_code,name,room_number,age,sex,ward_id,"
                    "status,diagnosis,admission_date,sedative_use) VALUES(?,?,?,?,?,?,?,?,?,?)",
                    (f["patient_code"].strip(), f["name"].strip(), f["room_number"].strip(),
                     int(f["age"]), f["sex"], int(f["ward_id"]),
                     f.get("status", "입원중"), f.get("diagnosis", ""),
                     f.get("admission_date", ""), 1 if f.get("sedative_use") else 0))
                new_pid = db.execute("SELECT last_insert_rowid()").fetchone()[0]
                nurse_id = f.get("nurse_id", "")
                if nurse_id:
                    db.execute(
                        "INSERT OR IGNORE INTO patient_assignments(user_id,patient_id) VALUES(?,?)",
                        (int(nurse_id), new_pid))
            flash("환자가 추가되었습니다.", "success")
            return redirect(url_for("admin_patients"))
        except Exception as e:
            flash(f"오류: {e}", "error")
    return render_template("admin_patient_form.html",
                           patient=None, default_code=default_code,
                           wards=wards, nurses=nurses, assigned_nurse_id=None)

@app.route("/admin/patients/<int:patient_id>/edit", methods=["GET", "POST"])
@admin_required
def admin_patient_edit(patient_id):
    with connect_db() as db:
        patient  = db.execute("SELECT * FROM patients WHERE id=?", (patient_id,)).fetchone()
        if not patient:
            abort(404)
        wards    = db.execute("SELECT * FROM wards ORDER BY id").fetchall()
        nurses   = db.execute("SELECT * FROM users WHERE role='NURSE' ORDER BY name").fetchall()
        assigned = db.execute(
            "SELECT user_id FROM patient_assignments WHERE patient_id=?",
            (patient_id,)).fetchone()
    if request.method == "POST":
        f = request.form
        try:
            with connect_db() as db:
                db.execute(
                    "UPDATE patients SET name=?,room_number=?,age=?,sex=?,ward_id=?,"
                    "status=?,diagnosis=?,admission_date=?,sedative_use=? WHERE id=?",
                    (f["name"].strip(), f["room_number"].strip(), int(f["age"]),
                     f["sex"], int(f["ward_id"]), f.get("status", "입원중"),
                     f.get("diagnosis", ""), f.get("admission_date", ""),
                     1 if f.get("sedative_use") else 0, patient_id))
                db.execute("DELETE FROM patient_assignments WHERE patient_id=?", (patient_id,))
                nurse_id = f.get("nurse_id", "")
                if nurse_id:
                    db.execute(
                        "INSERT INTO patient_assignments(user_id,patient_id) VALUES(?,?)",
                        (int(nurse_id), patient_id))
            flash("환자 정보가 수정되었습니다.", "success")
            return redirect(url_for("admin_patients"))
        except Exception as e:
            flash(f"오류: {e}", "error")
    return render_template("admin_patient_form.html",
                           patient=patient, default_code=None,
                           wards=wards, nurses=nurses,
                           assigned_nurse_id=assigned["user_id"] if assigned else None)

@app.route("/admin/patients/<int:patient_id>/delete", methods=["POST"])
@admin_required
def admin_patient_delete(patient_id):
    with connect_db() as db:
        patient = db.execute("SELECT * FROM patients WHERE id=?", (patient_id,)).fetchone()
        if not patient:
            abort(404)
        db.execute("DELETE FROM patient_assignments WHERE patient_id=?", (patient_id,))
        db.execute("DELETE FROM assessments WHERE patient_id=?", (patient_id,))
        db.execute("DELETE FROM patients WHERE id=?", (patient_id,))
    flash(f"'{patient['name']}' 환자를 삭제했습니다.", "success")
    return redirect(url_for("admin_patients"))

# ── 통계 ──────────────────────────────────────────────
@app.route("/stats")
@login_required
def stats():
    user = get_current_user()
    is_admin = user["role"] == "ADMIN"
    # 간호사는 자기 병동 집계만, 환자 이름은 담당 환자만 본다.
    pf = "" if is_admin else " AND p.ward_id=?"
    pa = () if is_admin else (user["ward_id"],)
    base = " FROM assessments a JOIN patients p ON p.id=a.patient_id WHERE 1=1" + pf
    with connect_db() as db:
        total_p  = db.execute("SELECT COUNT(*) FROM patients p WHERE 1=1" + pf, pa).fetchone()[0]
        total_a  = db.execute("SELECT COUNT(*)" + base, pa).fetchone()[0]
        recent_7 = db.execute(
            "SELECT COUNT(*)" + base + " AND a.assessed_at>=datetime('now','-7 days')", pa).fetchone()[0]
        cam_dist = {r["cam_result"]: r["cnt"] for r in db.execute(
            "SELECT a.cam_result, COUNT(*) cnt" + base + " GROUP BY a.cam_result", pa).fetchall()}
        rass_d   = db.execute(
            "SELECT a.rass_score, COUNT(*) cnt" + base +
            " GROUP BY a.rass_score ORDER BY a.rass_score", pa).fetchall()
        weekly   = db.execute(
            "SELECT strftime('%Y-%m-%d',a.assessed_at,'weekday 0','-6 days') AS wk,"
            " COUNT(*) total, SUM(a.cam_result='POSITIVE') positive" + base +
            " AND a.assessed_at>=datetime('now','-84 days')"
            " GROUP BY wk ORDER BY wk", pa).fetchall()
        ward_s   = db.execute(
            "SELECT w.name, COUNT(DISTINCT p.id) patients,"
            " COUNT(a.id) assessments,"
            " SUM(CASE WHEN a.cam_result='POSITIVE' THEN 1 ELSE 0 END) positive"
            " FROM wards w JOIN patients p ON p.ward_id=w.id"
            " LEFT JOIN assessments a ON a.patient_id=p.id"
            " WHERE 1=1" + pf + " GROUP BY w.id ORDER BY w.id", pa).fetchall()
        if is_admin:
            top_pos = db.execute(
                "SELECT p.name, p.room_number, COUNT(*) cnt"
                " FROM assessments a JOIN patients p ON p.id=a.patient_id"
                " WHERE a.cam_result='POSITIVE'"
                " GROUP BY a.patient_id ORDER BY cnt DESC LIMIT 10").fetchall()
        else:
            top_pos = db.execute(
                "SELECT p.name, p.room_number, COUNT(*) cnt"
                " FROM assessments a JOIN patients p ON p.id=a.patient_id"
                " JOIN patient_assignments pa ON pa.patient_id=p.id AND pa.user_id=?"
                " WHERE a.cam_result='POSITIVE'"
                " GROUP BY a.patient_id ORDER BY cnt DESC LIMIT 10",
                (user["id"],)).fetchall()
    pos_rate = round(cam_dist.get("POSITIVE", 0) / total_a * 100, 1) if total_a else 0
    chart = json.dumps({
        "cam":    cam_dist,
        "rass":   {"labels": [r["rass_score"] for r in rass_d],
                   "data":   [r["cnt"] for r in rass_d]},
        "weekly": {"labels":   [r["wk"] for r in weekly],
                   "total":    [r["total"] for r in weekly],
                   "positive": [r["positive"] or 0 for r in weekly]},
        "wards":  {"labels":      [r["name"] for r in ward_s],
                   "patients":    [r["patients"] for r in ward_s],
                   "assessments": [r["assessments"] for r in ward_s]},
    })
    scope = "전체 병동" if is_admin else user["ward_name"]
    return render_template("stats.html", scope=scope, is_admin=is_admin,
                           total_p=total_p, total_a=total_a,
                           recent_7=recent_7, pos_rate=pos_rate,
                           chart=chart, ward_stats=ward_s, top_pos=top_pos)

# ── 대시보드 ──────────────────────────────────────────
@app.route("/")
@nurse_required
def dashboard():
    sort = request.args.get("sort", "room")
    if sort not in SORT_OPTIONS:
        sort = "room"
    patients = get_patients_sorted(session["user_id"], sort)
    return render_template("dashboard.html", patients=patients,
                           sort=sort, sort_options=SORT_OPTIONS)

# ── PWA 서비스 워커 ───────────────────────────────────
@app.route("/sw.js")
def service_worker():
    response = send_from_directory(app.static_folder, "sw.js")
    response.headers["Cache-Control"] = "no-cache"
    return response

# ── 근무 인계 요약 ────────────────────────────────────
REASSESS_HOURS = {"POSITIVE": 4, "NEGATIVE": 8}

def format_kst(utc_dt):
    return (utc_dt + timedelta(hours=9)).strftime("%m-%d %H:%M")

def build_handover_item(patient, rows):
    item = {"patient": patient, "latest": rows[0] if rows else None,
            "flags": [], "priority": 9, "assessed_text": "—", "due_text": "—"}
    if not rows:
        item["flags"].append("미평가")
        item["priority"] = 3
        return item

    latest = rows[0]
    previous = rows[1] if len(rows) > 1 else None
    assessed_at = datetime.strptime(latest["assessed_at"][:19], "%Y-%m-%d %H:%M:%S")
    item["assessed_text"] = format_kst(assessed_at)

    hours = REASSESS_HOURS.get(latest["cam_result"])
    if hours:
        due_at = assessed_at + timedelta(hours=hours)
        item["due_text"] = format_kst(due_at)
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        if due_at < now:
            item["flags"].append("재평가 마감 초과")
            item["priority"] = 0

    if previous:
        if latest["cam_result"] == "POSITIVE" and previous["cam_result"] != "POSITIVE":
            item["flags"].append("POSITIVE 전환")
            item["priority"] = min(item["priority"], 1)
        elif latest["cam_result"] != "POSITIVE" and previous["cam_result"] == "POSITIVE":
            item["flags"].append("POSITIVE 해제")
        change = latest["rass_score"] - previous["rass_score"]
        if abs(change) >= 2:
            item["flags"].append(f"RASS {previous['rass_score']:+d} → {latest['rass_score']:+d}")
            item["priority"] = min(item["priority"], 2)

    if latest["cam_result"] == "POSITIVE":
        item["priority"] = min(item["priority"], 2)
    return item

@app.route("/handover")
@nurse_required
def handover():
    with connect_db() as db:
        patients = db.execute(
            "SELECT p.* FROM patients p "
            "JOIN patient_assignments pa ON pa.patient_id=p.id "
            "WHERE pa.user_id=? AND p.status='입원중' ORDER BY p.room_number",
            (session["user_id"],)).fetchall()
        items = []
        for patient in patients:
            rows = db.execute(
                "SELECT rass_score, cam_result, assessed_at, memo FROM assessments "
                "WHERE patient_id=? ORDER BY assessed_at DESC, id DESC LIMIT 2",
                (patient["id"],)).fetchall()
            items.append(build_handover_item(patient, rows))
    items.sort(key=lambda item: item["priority"])
    return render_template("handover.html", items=items)

# ── 환자 상세 ─────────────────────────────────────────
@app.route("/patients/<int:patient_id>")
@login_required
def patient_detail(patient_id):
    patient = get_assigned_patient(patient_id)
    if not patient:
        abort(403)
    with connect_db() as db:
        records = db.execute(
            "SELECT a.*, u.name AS nurse_name FROM assessments a"
            " JOIN users u ON u.id=a.nurse_id"
            " WHERE a.patient_id=? ORDER BY a.id DESC",
            (patient_id,)).fetchall()
    r_list = list(reversed(list(records)))
    cm = {"POSITIVE": "#d64545", "NEGATIVE": "#23865f", "UNASSESSABLE": "#654d96"}
    chart = json.dumps({
        "labels":  [r["assessed_at"][:16] for r in r_list],
        "rass":    [r["rass_score"] for r in r_list],
        "colors":  [cm.get(r["cam_result"], "#6b7b8d") for r in r_list],
        "results": [r["cam_result"] for r in r_list],
    })
    return render_template("patient_detail.html",
                           patient=patient, records=records, chart_data=chart)

# ── 환자 상태 수정 ────────────────────────────────────
@app.route("/patients/<int:patient_id>/status", methods=["POST"])
@login_required
def update_patient_status(patient_id):
    patient = get_assigned_patient(patient_id)
    if not patient:
        abort(403)
    status = request.form.get("status", "")
    if status not in ("입원중", "외출", "퇴원"):
        abort(400)
    with connect_db() as db:
        db.execute("UPDATE patients SET status=? WHERE id=?", (status, patient_id))
    flash(f"{patient['name']} 상태를 '{status}'로 변경했습니다.", "success")
    return redirect(request.referrer or url_for("mypage"))

# ── 평가 ──────────────────────────────────────────────
@app.route("/patients/<int:patient_id>/assess", methods=["GET", "POST"])
@nurse_required
def assessment(patient_id):
    patient = get_assigned_patient(patient_id)
    if not patient:
        abort(403)
    suggestion = None
    form_data  = {}
    with connect_db() as db:
        prev = db.execute(
            "SELECT * FROM assessments WHERE patient_id=? ORDER BY id DESC LIMIT 1",
            (patient_id,)).fetchone()
    if request.method == "POST":
        form_data = {key: request.form.get(key, "") for key in
                     ("observation_index", "narrative", "rass_score", "memo",
                      "feature1", "feature2", "feature4")}
        narrative = form_data["narrative"].strip()
        action = request.form.get("action", "save")

        if action == "suggest":
            suggestion = suggest_rass_ai(narrative)
            # 저장할 때 같은 서술에 대한 AI 결과인지 서버에서 확인하기 위해 세션에 보관
            session["ai_suggestion"] = {"patient_id": patient_id, "narrative": narrative,
                                        **suggestion}
            log_action(session["user_id"], "AI_SUGGEST",
                       f"patient_id={patient_id} status={suggestion['status']}")
            return render_template("assessment.html", patient=patient,
                                   suggestion=suggestion, form_data=form_data,
                                   obs_options=OBSERVATION_OPTIONS, prev=prev)

        saved_ai = session.get("ai_suggestion") or {}
        if saved_ai.get("patient_id") == patient_id and saved_ai.get("narrative") == narrative and narrative:
            suggestion = saved_ai
        try:
            obs_str = form_data["observation_index"]
            if not obs_str.isdigit() or int(obs_str) >= len(OBSERVATION_OPTIONS):
                raise ValueError("표준 절차 관찰 결과를 선택해 주세요.")
            obs_text = OBSERVATION_OPTIONS[int(obs_str)][0]
            if not form_data["rass_score"]:
                raise ValueError("최종 RASS 점수를 선택해 주세요.")
            rass = int(form_data["rass_score"])
            memo = form_data["memo"].strip()
            if rass <= -4:
                cam_result = calculate_cam_icu(rass)
                db_feat    = (None, None, None, None)
            else:
                for i in (1, 2, 4):
                    if form_data[f"feature{i}"] not in {"yes", "no"}:
                        raise ValueError("CAM-ICU 특징 F1, F2, F4를 모두 선택해 주세요.")
                f1, f2, f4 = (form_data[f"feature{i}"] == "yes" for i in (1, 2, 4))
                db_feat    = (f1, f2, derive_f3(rass), f4)
                cam_result = calculate_cam_icu(rass, f1, f2, f4)
            ai_status = suggestion["status"] if suggestion else "NOT_RUN"
            ai_score  = suggestion["score"] if suggestion else None
            ai_reason = suggestion["reason"] if suggestion else ""
            ai_evid   = suggestion.get("evidence", "") if suggestion else ""
            with connect_db() as db:
                cur = db.execute(
                    "INSERT INTO assessments(patient_id,nurse_id,observation,narrative,"
                    "ai_status,ai_rass,ai_reason,ai_evidence,"
                    "rass_score,feature1,feature2,feature3,feature4,cam_result,memo)"
                    " VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (patient_id, session["user_id"], obs_text, narrative,
                     ai_status, ai_score, ai_reason, ai_evid,
                     rass, *db_feat, cam_result, memo))
                new_assessment_id = cur.lastrowid
            session.pop("ai_suggestion", None)
            log_action(session["user_id"], "ASSESSMENT_CREATED",
                       f"assessment_id={new_assessment_id}")
            return redirect(url_for("result", assessment_id=new_assessment_id))
        except (ValueError, TypeError) as e:
            flash(str(e), "error")
    return render_template("assessment.html", patient=patient,
                           suggestion=suggestion, form_data=form_data,
                           obs_options=OBSERVATION_OPTIONS, prev=prev)

# ── 결과 ──────────────────────────────────────────────
@app.route("/results/<int:assessment_id>")
@login_required
def result(assessment_id):
    with connect_db() as db:
        row = db.execute(
            "SELECT a.*, p.name AS patient_name, p.room_number, u.name AS nurse_name"
            " FROM assessments a"
            " JOIN patients p ON p.id=a.patient_id"
            " JOIN users u ON u.id=a.nurse_id"
            " JOIN patient_assignments pa ON pa.patient_id=p.id"
            " WHERE a.id=? AND pa.user_id=?",
            (assessment_id, session["user_id"])).fetchone()
    if not row:
        abort(403)
    return render_template("result.html", row=row)

# ── 마이페이지 ────────────────────────────────────────
@app.route("/mypage")
@nurse_required
def mypage():
    sort = request.args.get("sort", "room")
    if sort not in SORT_OPTIONS:
        sort = "room"
    with connect_db() as db:
        stats = db.execute(
            "SELECT COUNT(*) total,"
            " SUM(cam_result='POSITIVE') positive,"
            " SUM(cam_result='NEGATIVE') negative,"
            " SUM(cam_result='UNASSESSABLE') unassessable"
            " FROM assessments WHERE nurse_id=?",
            (session["user_id"],)).fetchone()
    patients = get_patients_sorted(session["user_id"], sort)
    return render_template("mypage.html", stats=stats, patients=patients,
                           sort=sort, sort_options=SORT_OPTIONS)

# ── 에러 핸들러 ───────────────────────────────────────
@app.errorhandler(403)
def e403(_):
    return render_template("error.html", code=403, message="접근 권한이 없습니다."), 403

@app.errorhandler(404)
def e404(_):
    return render_template("error.html", code=404, message="페이지를 찾을 수 없습니다."), 404

@app.errorhandler(Exception)
def e500(e):
    print("\n" + "=" * 60, flush=True)
    print(f"[ERROR] {type(e).__name__}: {e}", flush=True)
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()
    print("=" * 60 + "\n", flush=True)
    return render_template("error.html", code=500,
                           message=f"서버 오류: {type(e).__name__}: {e}"), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 5000)), debug=False)
''')


In [ ]:
write('colab_start.py', r'''"""Colab 셀에서 `%run /content/cam_icu_rass_v11/colab_start.py`로 실행."""
import os, re, subprocess, sys, time, threading
from pathlib import Path

PROJECT = Path(__file__).resolve().parent
PORT = 8000

# ── Google Drive 연동 (DB 영구 보존) ──────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DB_DIR = Path("/content/drive/MyDrive/KY_CAM_ICU_v11")
    DB_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAM_ICU_DB_PATH"] = str(DB_DIR / "cam_icu.db")
    print(f"✅ Google Drive 연동 → {os.environ['CAM_ICU_DB_PATH']}")
except Exception as e:
    print(f"⚠️  Drive 연동 실패 ({e}) → 로컬 임시 저장으로 진행")

# ── AI API 키 (Colab 왼쪽 🔑 Secrets) ─────────────────
try:
    from google.colab import userdata
    for key_name in ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "AI_PROVIDER", "AI_MODEL"):
        try:
            value = userdata.get(key_name)
        except Exception:
            value = None
        if value:
            os.environ[key_name] = value
    found = [k for k in ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY") if os.environ.get(k)]
    print(f"🤖 AI 키: {', '.join(found) if found else '없음 → AI 해석은 사용 불가로 표시됩니다'}")
except ImportError:
    pass

# 프로젝트 디렉터리를 임포트 경로 최상위에 추가
sys.path.insert(0, str(PROJECT))
os.chdir(str(PROJECT))

# 패키지 설치
print("📦 패키지 설치 중...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "Flask>=3.0.0", "Werkzeug>=3.0.0", "requests>=2.28.0"],
    check=True,
)
print("✅ 설치 완료")

# 앱 로드 — 실패 시 traceback 출력
print("🔍 앱 초기화 중...")
try:
    from app import app as flask_app
    print("✅ 앱 초기화 성공")
except Exception:
    import traceback
    print("❌ 앱 초기화 실패 — traceback:")
    traceback.print_exc()
    raise

# Flask 내장 서버(threaded)를 백그라운드 스레드로 실행
# → 오류 발생 시 traceback이 이 셀에 바로 출력됨
def _run():
    flask_app.run(host="0.0.0.0", port=PORT, debug=False,
                  use_reloader=False, threaded=True)

threading.Thread(target=_run, daemon=True).start()

# 서버 준비 대기
import requests as _req
print("🚀 서버 시작 중...")
for _ in range(30):
    try:
        if _req.get(f"http://127.0.0.1:{PORT}/login", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("서버 시작 실패 — 위 traceback을 확인하세요.")

print("✅ 서버 준비 완료")

# cloudflared 공개 링크 생성
binary = Path("/usr/local/bin/cloudflared")
if not binary.exists():
    print("🌐 cloudflared 다운로드 중...")
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", str(binary)],
        check=True,
    )
    binary.chmod(0o755)

tunnel = subprocess.Popen(
    [str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    m = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        time.sleep(3)
        print("\n✅ 접속 링크:", m.group(0))
        print("   계정: nurse01 / 1234  또는  nurse02 / 1234")
        break
else:
    raise RuntimeError("공개 링크 생성 실패")
''')


In [ ]:
write('eval_ai.py', r'''"""AI RASS 해석 정확도 평가.

사용법:  python eval_ai.py [사례파일.csv]
CSV 열: narrative, expected  (expected가 비어 있으면 '판단 보류'가 정답)
API 키가 설정돼 있어야 한다. 비교용으로 기존 키워드 방식 결과도 함께 출력한다.

⚠️ tests/ai_cases.csv는 형식 예시다. 발표용 평가는 팀원이 직접 쓴 사례로 하세요.
   (AI가 만든 문제로 AI를 채점하면 공정한 평가가 아닙니다.)
"""
import csv
import sys

from ai_rass import suggest_rass_ai
from cam_icu_logic import suggest_rass


def main(path="tests/ai_cases.csv"):
    with open(path, encoding="utf-8") as f:
        cases = list(csv.DictReader(f))
    ai_ok = kw_ok = abstain = errors = 0
    for i, case in enumerate(cases, 1):
        expected = int(case["expected"]) if case["expected"].strip() else None
        ai = suggest_rass_ai(case["narrative"])
        kw = suggest_rass(case["narrative"])["score"]
        if ai["status"] in ("UNAVAILABLE", "ERROR"):
            errors += 1
        if ai["status"] == "ABSTAIN":
            abstain += 1
        ai_ok += ai["score"] == expected
        kw_ok += kw == expected
        mark = "O" if ai["score"] == expected else "X"
        print(f"{i:2d}. [{mark}] 정답 {expected}  AI {ai['score']}  키워드 {kw}  | {case['narrative']}")
        if ai["status"] != "SCORED":
            print(f"      └ {ai['status']}: {ai['reason']}")
    n = len(cases)
    print(f"\nAI 정확도      {ai_ok}/{n} ({ai_ok / n * 100:.1f}%)  · 판단 보류 {abstain}건 · 오류 {errors}건")
    print(f"키워드 정확도  {kw_ok}/{n} ({kw_ok / n * 100:.1f}%)")


if __name__ == "__main__":
    main(*sys.argv[1:])
''')


In [ ]:
write('COLAB_GUIDE.md', r'''# Colab 실행 순서

## 1. ZIP 업로드와 압축 해제
```python
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')
```

## 2. DB를 Google Drive에 영구 보관
```python
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
db_dir = Path('/content/drive/MyDrive/cam_icu_rass_v2_data')
db_dir.mkdir(parents=True, exist_ok=True)
os.environ['CAM_ICU_DB_PATH'] = str(db_dir / 'cam_icu.db')
os.environ['SECRET_KEY'] = '원하는-긴-임의문자열'
```

## 3. 앱과 공개 링크 실행
```python
%run /content/cam_icu_rass_v2/colab_start.py
```

출력된 `trycloudflare.com` 주소로 접속합니다. Colab 런타임이 종료되면 링크는 꺼지지만, 평가 데이터는 Google Drive의 DB에 남습니다. 다음 실행 때 1~3단계를 반복하면 같은 데이터가 연결됩니다.

## 주의
- `colab_start.py`는 한 런타임에서 한 번만 실행합니다.
- 코드 수정 후에는 서버를 재시작해야 반영됩니다.
- 영구적으로 켜진 주소가 필요하면 Render 등 별도 서버에 배포해야 합니다.
''')


In [ ]:
write('README.md', r'''# CAM-ICU/RASS 디지털 섬망 평가 시스템 v2

교육·시연용 Flask 프로젝트입니다. 실제 임상 진단을 대신하지 않습니다.

## 주요 기능
- 간호사 로그인과 담당 병동 기반 접근 제한
- 환자 목록, 상세 기록, 간호사 마이페이지
- 표준 관찰 기록 기반 시연용 AI RASS 참고 제안
- 간호사가 최종 RASS 확정
- RASS -4/-5 게이트 및 CAM-ICU 서버 계산
- 공식: F1 AND F2 AND (F3 OR F4)
- 감사 로그와 SQLite 저장

## 시연 계정
- `nurse01 / 1234` (중환자실 A, 환자 3명)
- `nurse02 / 1234` (중환자실 B, 환자 2명)

## 로컬 실행
```bash
python -m pip install -r requirements.txt
python app.py
```

## Colab 실행
ZIP을 `/content`에 업로드·압축 해제한 뒤 `colab_start.py`를 실행합니다.
Google Drive에 DB를 보존하려면 먼저 Drive를 연결하고 `CAM_ICU_DB_PATH` 환경변수를 Drive 경로로 지정합니다.

## v11 변경 사항
- **AI 관찰 해석:** 간호사가 쓴 관찰 서술을 LLM이 RASS 공식 정의에 맞춰 해석하고, 제안 점수와 서술 원문 인용을 보여줍니다. 인용이 원문에 없으면 점수를 버립니다(환각 차단). 근거가 부족하면 "판단 보류"로 답합니다. 최종 RASS는 간호사가 확정하며, AI 제안과 다르면 경고가 뜹니다.
- **통계 권한:** 간호사는 소속 병동 집계만 보고, POSITIVE 상위 환자 이름은 담당 환자만 보입니다. 관리자는 전체를 봅니다.
- 이전 "AI 참고 점수"(표에서 점수를 찾던 기능)는 제거했습니다. 2단계 관찰을 고르면 최종 RASS 칸이 자동으로 채워집니다.
- 가상 평가 기록은 AI를 실행한 적이 없으므로 `NOT_RUN`으로 저장됩니다.

## AI 설정
키 하나만 있으면 됩니다.

| 서비스 | 키 이름 | 기본 모델 |
|---|---|---|
| Google Gemini | `GEMINI_API_KEY` | `gemini-2.5-flash` |
| OpenAI | `OPENAI_API_KEY` | `gpt-4.1-mini` |
| Anthropic | `ANTHROPIC_API_KEY` | `claude-haiku-4-5` |

- **Colab:** 왼쪽 🔑(Secrets)에 키 이름과 값을 추가하고 "노트북 액세스"를 켭니다.
- **PythonAnywhere:** WSGI 파일에서 `os.environ["GEMINI_API_KEY"] = "..."`처럼 설정합니다.
- 모델 이름이 바뀌었으면 `AI_MODEL`로 지정합니다. 여러 키가 있으면 `AI_PROVIDER`로 고릅니다.
- **API 키를 코드나 GitHub에 절대 넣지 마세요.**
- 키가 없으면 앱은 정상 동작하고, AI 해석만 "사용 불가"로 표시됩니다.

## AI 정확도 평가
```bash
python eval_ai.py tests/ai_cases.csv
```
`tests/ai_cases.csv`는 형식 예시입니다. 발표용 평가는 팀원이 정답 RASS를 붙여 직접 쓴 사례로 하세요. 기존 키워드 방식과의 정확도 비교도 함께 출력됩니다.
''')


In [ ]:
write('static/manifest.webmanifest', r'''{
  "name": "KY CAM-ICU",
  "short_name": "CAM-ICU",
  "start_url": "/",
  "scope": "/",
  "display": "standalone",
  "background_color": "#f3f6f7",
  "theme_color": "#0f6e86",
  "icons": [
    {"src": "/static/icons/icon-192.png", "sizes": "192x192", "type": "image/png"},
    {"src": "/static/icons/icon-512.png", "sizes": "512x512", "type": "image/png"}
  ]
}
''')


In [ ]:
write('static/style.css', r''':root{--blue:#3979d8;--navy:#17365d;--bg:#f4f8fc;--line:#dbe5ef;--text:#203044;--muted:#6b7b8d;--red:#d64545;--green:#23865f}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--text);font-family:"Malgun Gothic","Apple SD Gothic Neo",sans-serif}

/* ── 헤더 ── */
header{height:56px;background:white;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 max(24px,calc((100% - 1200px)/2))}
.brand{font-weight:800;color:var(--navy);font-size:18px;text-decoration:none}
nav{display:flex;gap:18px;align-items:center;font-size:14px}
nav a,.back{color:var(--blue);text-decoration:none}

/* ── 정보 바 ── */
.info-bar{background:#eef4ff;border-bottom:1px solid #d0e0f8;padding:6px max(24px,calc((100% - 1200px)/2));display:flex;gap:20px;font-size:13px;color:var(--navy);font-weight:600}
.info-bar span{display:flex;align-items:center;gap:4px}

/* ── 레이아웃 ── */
main{max-width:1200px;margin:24px auto;padding:0 20px;min-height:calc(100vh - 160px)}
footer{text-align:center;color:var(--muted);font-size:12px;padding:20px}

/* ── 카드 ── */
.card{background:white;border:1px solid var(--line);border-radius:14px;padding:22px}
.login-card,.result-card{max-width:520px;margin:55px auto;background:white;border:1px solid var(--line);border-radius:18px;padding:32px;box-shadow:0 12px 32px #284d7812}

/* ── 툴바 (검색+정렬) ── */
.toolbar{display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:12px;margin-bottom:14px}
.search-input{padding:9px 14px;border:1px solid var(--line);border-radius:24px;width:260px;font:inherit;font-size:14px}
.search-input:focus{outline:none;border-color:var(--blue)}
.sort-btns{display:flex;gap:6px;flex-wrap:wrap}
.sort-btn{padding:6px 13px;border:1px solid var(--line);border-radius:20px;font-size:13px;text-decoration:none;color:var(--muted);background:white}
.sort-btn.active{background:var(--blue);color:white;border-color:var(--blue)}

/* ── 테이블 ── */
.table-wrap{overflow-x:auto}
table{border-collapse:collapse;width:100%;min-width:700px}
th,td{text-align:left;border-bottom:1px solid var(--line);padding:11px 10px;font-size:14px}
th{font-size:13px;color:var(--muted);font-weight:600}
td a{color:var(--blue);text-decoration:none;font-weight:600}
.room{font-weight:800;color:var(--navy)}

/* ── 카운트다운 ── */
.countdown{font-size:13px;font-weight:700;padding:3px 8px;border-radius:6px}
.countdown.ok{color:#19704e;background:#e2f6ed}
.countdown.warn{color:#a07010;background:#fff8e1}
.countdown.urgent{color:#c0392b;background:#ffe5e5}
.countdown.overdue{color:#c0392b;background:#ffd6d6;animation:pulse 1.5s infinite}
.countdown.special{color:#654d96;background:#f3eeff;font-size:12px}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.6}}

/* ── 상태 태그 ── */
.status-tag{font-size:12px;font-weight:700;padding:3px 9px;border-radius:20px}
.s-in{background:#e2f6ed;color:#19704e}
.s-out{background:#fff8e1;color:#a07010}
.s-dis{background:#f0f4f8;color:#526476}
.row-discharged td{opacity:.45}
.row-out{background:#fffdf2}

/* ── 배지 ── */
.badge{display:inline-block;border-radius:999px;padding:4px 9px;font-size:12px;font-weight:800}
.positive{background:#ffe5e5;color:#a92f2f}
.negative{background:#e2f6ed;color:#19704e}
.unassessable{background:#eee9f8;color:#654d96}
.neutral{background:#edf1f5;color:#526476}

/* ── 페이지 헤더 ── */
.page-head{display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:16px;flex-wrap:wrap;gap:10px}
.page-head p,.muted{color:var(--muted)}
h1{color:var(--navy);margin:4px 0 8px}
h2{margin-top:0;color:var(--navy)}

/* ── 버튼 ── */
button,.button{border:0;border-radius:9px;background:var(--blue);color:white;padding:10px 16px;font-weight:700;text-decoration:none;cursor:pointer;display:inline-block;font:inherit}
.login-card button{width:100%;margin-top:14px}
.secondary{background:#eef4fb;color:var(--blue);border:1px solid #cbdcf0}
.btn-sm{padding:5px 11px;font-size:13px}
.btn-danger{background:#fff0f0;color:#c0392b;border:1px solid #f5c6c6;border-radius:7px;padding:5px 12px;font-size:13px;font-weight:700;cursor:pointer}
.btn-danger:hover{background:#c0392b;color:white}
.submit{width:100%;margin-top:14px}

/* ── 폼 ── */
label{display:block;font-weight:700;margin:14px 0 5px}
input,select{width:100%;padding:11px;border:1px solid #bdcbd9;border-radius:9px;background:white;font:inherit}

/* ── 알림 ── */
.alert{max-width:900px;padding:12px;border-radius:9px;margin-bottom:14px}
.alert.error{background:#ffeaea;color:#9b2e2e}
.alert.success{background:#e2f6ed;color:#19704e}

/* ── 로그인 ── */
.demo{margin-top:18px;padding:14px;background:#eef6ff;border-radius:10px;color:#355574;font-size:14px}

/* ── 마이페이지 ── */
.profile{display:flex;gap:18px;align-items:center}
.avatar{width:58px;height:58px;border-radius:50%;display:grid;place-items:center;background:#dbeaff;color:var(--blue);font-size:24px;font-weight:800}
.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:14px 0 20px}
.stats div{background:white;border:1px solid var(--line);border-radius:12px;padding:16px}
.stats b{font-size:26px;color:var(--navy);display:block}
.stats span{color:var(--muted);font-size:13px}

/* ── 정렬 바 (마이페이지) ── */
.sort-bar{display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:14px}
.sort-label{font-weight:700;color:var(--navy)}

/* ── 상태 수정 드롭다운 ── */
.status-form{margin:0}
.status-sel{width:auto;padding:4px 8px;font-size:12px;border-radius:7px;font-weight:600;cursor:pointer;border:1px solid}
.status-in{background:#e2f6ed;border-color:#23865f;color:#19704e}
.status-out{background:#fff8e1;border-color:#e6a817;color:#a07010}
.status-discharged{background:#f0f4f8;border-color:#8a9ab0;color:#526476}

/* ── 평가 스텝 ── */
.assessment-form{display:grid;gap:16px;max-width:780px}
.step-card{position:relative}
.step-title{display:flex;align-items:center;gap:10px;margin:0 0 14px;color:var(--navy);font-size:16px}
.step-num{display:inline-flex;align-items:center;justify-content:center;width:26px;height:26px;border-radius:50%;background:var(--blue);color:white;font-size:13px;font-weight:800;flex-shrink:0}
.obs-option{display:flex;align-items:center;gap:12px;padding:11px 14px;border:1.5px solid var(--line);border-radius:9px;margin-bottom:7px;cursor:pointer;font-weight:400;transition:border-color .15s,background .15s}
.obs-option input[type=radio]{width:17px;height:17px;flex-shrink:0;accent-color:var(--blue)}
.obs-option:hover,.obs-selected,.obs-option:has(input:checked){border-color:var(--blue);background:#eef4ff;font-weight:600}
.obs-label{line-height:1.4}
.radio{display:inline-flex;align-items:center;gap:6px;margin-right:18px;font-weight:400}
.radio input{width:auto}
fieldset{border:0;border-bottom:1px solid var(--line);padding:13px 0}
.disabled{opacity:.5;pointer-events:none}
.formula,.ai-box,.notice{margin-top:14px;padding:13px;background:#eef6ff;border-radius:9px;color:#355574;font-size:14px}
.ai-box{border-left:4px solid var(--blue)}

/* ── 결과 화면 ── */
.result{text-align:center;font-size:34px;padding:16px;border-radius:12px}
.feature-row{display:flex;flex-wrap:wrap;gap:8px;margin-top:12px}
.feature-row span{background:#f0f4f8;padding:8px;border-radius:7px;font-size:14px}
dl{display:grid;grid-template-columns:90px 1fr;gap:10px;margin-top:20px}
dt{font-weight:800}
dd{margin:0}
.unassessable-msg{color:#654d96;font-weight:700;padding:10px 14px;background:#f3eeff;border-radius:9px;margin-top:8px}

/* ── 모달 ── */
.modal-overlay{position:fixed;inset:0;background:rgba(0,0,0,.45);display:flex;align-items:center;justify-content:center;z-index:1000}
.modal-box{background:white;border-radius:16px;padding:30px;max-width:420px;width:90%;box-shadow:0 20px 60px rgba(0,0,0,.25)}
.modal-box h3{margin:0 0 10px;color:var(--navy)}
.modal-btns{display:flex;gap:10px;margin-top:18px}
.modal-btns button{flex:1}

/* ── 그래프 ── */
.chart-legend{margin-top:10px;font-size:13px;color:var(--muted);display:flex;gap:14px;flex-wrap:wrap}
.legend-dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}

/* ── 시간 셀 ── */
.time-cell{font-size:13px;color:var(--muted);white-space:nowrap}

.empty{color:var(--muted);text-align:center;padding:30px}

@media(max-width:700px){
  header{height:auto;padding:12px 16px;flex-wrap:wrap;gap:8px}
  .info-bar{flex-wrap:wrap;gap:8px}
  .toolbar{flex-direction:column;align-items:stretch}
  .search-input{width:100%}
  .stats{grid-template-columns:repeat(2,1fr)}
  main{margin-top:16px;padding:0 12px}
}

/* ── 통계 ── */
.chart-grid{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-top:18px}
.chart-wide{grid-column:1/-1}
.chart-wrap{position:relative;height:250px}
.chart-wrap-wide{position:relative;height:220px}
.pagination{display:flex;justify-content:center;align-items:center;gap:14px;margin-top:18px}

/* ── 이전 평가 참고 박스 ── */
.prev-box{background:#f3eeff;border:1px solid #d4c2f7;border-radius:10px;padding:12px 16px;margin-bottom:16px;font-size:14px;color:#4a3575}

/* ── admin.html 수정 ── */
.admin-links{display:flex;gap:12px;margin-bottom:20px;flex-wrap:wrap}

/* ── 병동 탭 (v7) ── */
.ward-tabs{display:flex;gap:8px;margin-bottom:16px;flex-wrap:wrap}
.ward-tab{padding:8px 16px;border-radius:20px;text-decoration:none;font-size:13px;
  font-weight:700;color:var(--muted);background:white;border:1.5px solid var(--line);
  display:flex;align-items:center;gap:6px;transition:.15s}
.ward-tab:hover{border-color:var(--blue);color:var(--blue)}
.ward-tab.active{background:var(--blue);color:white;border-color:var(--blue)}
.ward-count{display:inline-flex;align-items:center;justify-content:center;
  min-width:22px;height:18px;padding:0 5px;border-radius:10px;font-size:11px;font-weight:800;
  background:rgba(255,255,255,0.25);color:inherit}
.ward-tab:not(.active) .ward-count{background:#eef4ff;color:var(--blue)}
.badge-ward{display:inline-block;padding:2px 8px;border-radius:6px;font-size:12px;
  font-weight:700;background:#dbeef8;color:var(--navy)}

/* ── 폴더 레이아웃 (v7) ── */
.folder-layout{display:flex;gap:20px;align-items:flex-start}
.folder-panel{width:180px;flex-shrink:0;background:white;border:1px solid var(--line);
  border-radius:14px;padding:14px;box-shadow:0 2px 8px rgba(44,110,138,.06);
  position:sticky;top:80px}
.folder-panel-title{font-size:11px;font-weight:800;color:var(--muted);
  letter-spacing:1px;text-transform:uppercase;margin-bottom:10px;padding:0 4px}
.folder-item{display:flex;align-items:center;gap:8px;padding:10px 10px;
  border-radius:9px;text-decoration:none;color:var(--text);font-size:14px;
  font-weight:600;transition:.15s;margin-bottom:4px}
.folder-item:hover{background:var(--bg)}
.folder-item.active{background:#dbeef8;color:var(--navy)}
.folder-item.active .folder-icon{filter:none}
.folder-icon{font-size:18px;flex-shrink:0}
.folder-name{flex:1;font-size:13px}
.folder-badge{background:#eef4ff;color:var(--blue);border-radius:10px;
  padding:2px 7px;font-size:11px;font-weight:800;flex-shrink:0}
.folder-item.active .folder-badge{background:var(--blue);color:white}
.folder-content{flex:1;min-width:0}
.folder-breadcrumb{font-size:15px;font-weight:800;color:var(--navy);
  margin-bottom:14px;display:flex;align-items:center;gap:8px}
@media(max-width:700px){
  .folder-layout{flex-direction:column}
  .folder-panel{width:100%;position:static;display:flex;flex-wrap:wrap;gap:6px;padding:10px}
  .folder-panel-title{width:100%}
  .folder-item{flex:none;padding:7px 12px}
}
''')


In [ ]:
write('static/sw.js', r'''// 환자 정보가 기기에 남지 않도록 어떤 응답도 캐시하지 않는다.
self.addEventListener('install', () => self.skipWaiting());
self.addEventListener('activate', event => event.waitUntil(self.clients.claim()));
self.addEventListener('fetch', event => event.respondWith(fetch(event.request)));
''')


In [ ]:
write('templates/admin.html', r'''{% extends 'base.html' %}
{% block title %}관리자 페이지{% endblock %}
{% block content %}
<div class="page-head">
  <div><h1>병동 관리</h1><p class="muted">간호사 계정 관리</p></div>
  <a class="button" href="{{ url_for('register_nurse') }}">+ 간호사 추가</a>
</div>

<section class="card">
  <h2>등록된 간호사</h2>
  {% if nurses %}
  <div class="table-wrap">
    <table>
      <thead><tr><th>이름</th><th>아이디</th><th>사번</th><th>병동</th><th>담당 환자</th><th>등록일</th><th></th></tr></thead>
      <tbody>
        {% for n in nurses %}
        <tr>
          <td><b>{{ n.name }}</b></td>
          <td>{{ n.username }}</td>
          <td>{{ n.employee_no }}</td>
          <td>{{ n.ward_name }}</td>
          <td>{{ n.patient_count }}명</td>
          <td>{{ n.created_at[:10] }}</td>
          <td>
            <button class="btn-danger" onclick="openModal({{ n.id }})">삭제</button>
          </td>
        </tr>

        <!-- 삭제 확인 모달 -->
        <div id="modal-{{ n.id }}" class="modal-overlay" style="display:none"
             onclick="if(event.target===this)closeModal({{ n.id }})">
          <div class="modal-box">
            <h3>간호사 계정 삭제</h3>
            <p><b>{{ n.name }}</b> 간호사 계정을 삭제하려면<br>
               해당 간호사의 아이디와 비밀번호를 입력하세요.</p>
            <form method="post" action="{{ url_for('delete_nurse', nurse_id=n.id) }}">
              <label>간호사 아이디
                <input name="confirm_username" required autocomplete="off"
                       placeholder="{{ n.username }}">
              </label>
              <label>간호사 비밀번호
                <input type="password" name="confirm_password" required autocomplete="off">
              </label>
              <div class="modal-btns">
                <button type="button" class="secondary" onclick="closeModal({{ n.id }})">취소</button>
                <button type="submit" class="btn-danger">삭제 확인</button>
              </div>
            </form>
          </div>
        </div>
        {% endfor %}
      </tbody>
    </table>
  </div>
  {% else %}
  <p class="empty">등록된 간호사가 없습니다.</p>
  {% endif %}
</section>

<script>
function openModal(id)  { document.getElementById('modal-' + id).style.display = 'flex'; }
function closeModal(id) { document.getElementById('modal-' + id).style.display = 'none'; }
</script>
{% endblock %}
''')


In [ ]:
write('templates/admin_patient_form.html', r'''{% extends 'base.html' %}
{% block title %}{% if patient %}환자 수정{% else %}환자 추가{% endif %}{% endblock %}
{% block content %}
<a class="back" href="{{ url_for('admin_patients') }}">← 환자 목록</a>
<section class="login-card" style="max-width:600px">
  <h1>{% if patient %}환자 정보 수정{% else %}새 환자 추가{% endif %}</h1>
  <form method="post">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
      <label>환자 코드<input name="patient_code" value="{{ patient.patient_code if patient else default_code }}" {% if patient %}readonly{% endif %} required></label>
      <label>이름<input name="name" value="{{ patient.name if patient else '' }}" required></label>
      <label>병실<input name="room_number" value="{{ patient.room_number if patient else '' }}" required></label>
      <label>나이<input name="age" type="number" min="0" max="120" value="{{ patient.age if patient else '' }}" required></label>
      <label>성별
        <select name="sex" required>
          <option value="남" {% if not patient or patient.sex=='남' %}selected{% endif %}>남</option>
          <option value="여" {% if patient and patient.sex=='여' %}selected{% endif %}>여</option>
        </select>
      </label>
      <label>병동
        <select name="ward_id" required>
          {% for w in wards %}
          <option value="{{ w.id }}" {% if patient and patient.ward_id==w.id %}selected{% endif %}>{{ w.name }}</option>
          {% endfor %}
        </select>
      </label>
      <label>상태
        <select name="status">
          {% for s in ['입원중','외출','퇴원'] %}
          <option {% if patient and patient.status==s %}selected{% endif %}>{{ s }}</option>
          {% endfor %}
        </select>
      </label>
      <label>담당 간호사
        <select name="nurse_id">
          <option value="">미배정</option>
          {% for n in nurses %}
          <option value="{{ n.id }}" {% if assigned_nurse_id and assigned_nurse_id==n.id %}selected{% endif %}>{{ n.name }} ({{ n.employee_no }})</option>
          {% endfor %}
        </select>
      </label>
    </div>
    <label>진단명<input name="diagnosis" value="{{ patient.diagnosis if patient else '' }}" placeholder="예: 폐렴, 패혈증"></label>
    <label>입원일<input type="date" name="admission_date" value="{{ patient.admission_date if patient else '' }}"></label>
    <label style="display:flex;align-items:center;gap:10px;font-weight:700;margin-top:14px">
      <input type="checkbox" name="sedative_use" style="width:auto" {% if patient and patient.sedative_use %}checked{% endif %}>
      진정제·진통제 사용 중
    </label>
    <button class="submit">{% if patient %}수정 저장{% else %}환자 추가{% endif %}</button>
  </form>
</section>
{% endblock %}
''')


In [ ]:
write('templates/admin_patients.html', r'''{% extends 'base.html' %}
{% block title %}환자 관리{% endblock %}
{% block content %}
<div class="page-head">
  <div><h1>환자 관리</h1></div>
  <a class="button" href="{{ url_for('admin_patient_new') }}">+ 환자 추가</a>
</div>

<div class="folder-layout">

  <!-- 왼쪽 폴더 패널 -->
  <aside class="folder-panel">
    <p class="folder-panel-title">병동 선택</p>

    <a class="folder-item{% if not ward_filter %} active{% endif %}"
       href="{{ url_for('admin_patients', q=q) }}">
      <span class="folder-icon">📁</span>
      <span class="folder-name">전체 환자</span>
      <span class="folder-badge">{{ total_count }}</span>
    </a>

    {% for w in wards %}
    <a class="folder-item{% if ward_filter == w.id|string %} active{% endif %}"
       href="{{ url_for('admin_patients', q=q, ward=w.id) }}">
      <span class="folder-icon">📂</span>
      <span class="folder-name">{{ w.name }}</span>
      <span class="folder-badge">{{ ward_counts.get(w.id, 0) }}</span>
    </a>
    {% endfor %}
  </aside>

  <!-- 오른쪽 콘텐츠 -->
  <div class="folder-content">

    <!-- 현재 위치 + 검색 -->
    <div class="folder-breadcrumb">
      <span>
        {% if ward_filter %}
          {% for w in wards %}{% if w.id|string == ward_filter %}📂 {{ w.name }}{% endif %}{% endfor %}
        {% else %}
          📁 전체 환자
        {% endif %}
      </span>
      <span class="muted">— {{ total }}명</span>
    </div>

    <form class="toolbar" method="get">
      {% if ward_filter %}<input type="hidden" name="ward" value="{{ ward_filter }}">{% endif %}
      <input name="q" value="{{ q }}" type="text"
             placeholder="🔍 이름·코드·병실 검색..." class="search-input">
      <button type="submit" class="secondary" style="padding:9px 16px">검색</button>
      {% if q %}
      <a href="{{ url_for('admin_patients', ward=ward_filter) }}"
         class="secondary" style="padding:9px 14px;border-radius:9px;text-decoration:none;display:inline-block">✕</a>
      {% endif %}
    </form>

    <!-- 환자 테이블 -->
    <div class="table-wrap card">
      <table>
        <thead>
          <tr><th>코드</th><th>이름</th><th>나이·성별</th><th>병실</th><th>병동</th>
              <th>진단명</th><th>담당 간호사</th><th>상태</th><th></th></tr>
        </thead>
        <tbody>
          {% for p in patients %}
          <tr>
            <td class="muted">{{ p.patient_code }}</td>
            <td><b>{{ p.name }}</b></td>
            <td>{{ p.age }}세 · {{ p.sex }}</td>
            <td>{{ p.room_number }}</td>
            <td><span class="badge-ward">{{ p.ward_name }}</span></td>
            <td>{{ p.diagnosis or '—' }}</td>
            <td>{{ p.nurse_name or '미배정' }}</td>
            <td><span class="status-tag
              {%- if p.status=='입원중' %} s-in
              {%- elif p.status=='외출' %} s-out
              {%- else %} s-dis{%- endif %}">{{ p.status }}</span></td>
            <td style="white-space:nowrap">
              <a class="secondary btn-sm"
                 href="{{ url_for('admin_patient_edit', patient_id=p.id) }}"
                 style="padding:5px 10px;border-radius:7px;text-decoration:none;display:inline-block">수정</a>
              <form method="post" action="{{ url_for('admin_patient_delete', patient_id=p.id) }}"
                    style="display:inline"
                    onsubmit="return confirm('{{ p.name }} 환자를 삭제하시겠습니까?\n평가 기록도 함께 삭제됩니다.')">
                <button class="btn-danger">삭제</button>
              </form>
            </td>
          </tr>
          {% endfor %}
          {% if not patients %}
          <tr><td colspan="9" class="empty">해당 조건의 환자가 없습니다.</td></tr>
          {% endif %}
        </tbody>
      </table>
    </div>

    <!-- 페이지네이션 -->
    {% if total_pages > 1 %}
    <div class="pagination">
      {% if page > 1 %}
      <a href="?q={{ q }}&ward={{ ward_filter }}&page={{ page-1 }}" class="sort-btn">← 이전</a>
      {% endif %}
      <span class="muted">{{ page }} / {{ total_pages }}페이지</span>
      {% if page < total_pages %}
      <a href="?q={{ q }}&ward={{ ward_filter }}&page={{ page+1 }}" class="sort-btn">다음 →</a>
      {% endif %}
    </div>
    {% endif %}

  </div><!-- /folder-content -->
</div><!-- /folder-layout -->
{% endblock %}
''')


In [ ]:
write('templates/assessment.html', r'''{% extends 'base.html' %}
{% block content %}
<a class="back" href="{{ url_for('patient_detail', patient_id=patient.id) }}">← 환자 정보</a>
<h1>{{ patient.name }} 평가</h1>

{% if prev %}
<div class="prev-box">
  <b>직전 평가 참고</b> &nbsp;
  {{ prev.assessed_at[:16] }} &nbsp;
  RASS {{ '%+d'|format(prev.rass_score) }} &nbsp;
  <span class="badge {{ prev.cam_result|lower }}">{{ prev.cam_result }}</span>
  {% if prev.memo %} &nbsp; 메모: {{ prev.memo }}{% endif %}
</div>
{% endif %}

<form method="post" class="assessment-form">
  <section class="card step-card">
    <h2 class="step-title"><span class="step-num">1</span> 관찰 서술 + AI 해석</h2>
    <p class="muted">본 그대로 적으세요. 어떤 자극에 어떻게 반응했는지가 들어가야 AI가 판단할 수 있습니다.</p>
    <textarea name="narrative" rows="3" maxlength="600" placeholder="예) 이름 부르니 눈 떴는데 5초쯤 보다가 다시 감음" style="width:100%;padding:10px;border:1px solid #bdcbd9;border-radius:9px;font:inherit;resize:vertical">{{ form_data.get('narrative','') }}</textarea>
    <p class="notice">⚠️ 서술은 외부 AI 서비스로 전송됩니다. 환자 이름·등록번호 등 개인정보를 쓰지 마세요.</p>
    <button class="secondary" name="action" value="suggest" type="submit" formnovalidate>AI 해석 요청</button>
    {% if suggestion %}
    <div class="ai-box" id="ai-box" data-score="{{ '' if suggestion.score is none else suggestion.score }}">
      {% if suggestion.score is not none %}<strong>AI 제안 RASS: {{ '%+d'|format(suggestion.score) }}</strong>
      {% else %}<strong>AI {{ {'ABSTAIN':'판단 보류','UNAVAILABLE':'사용 불가','ERROR':'오류'}.get(suggestion.status, suggestion.status) }}</strong>{% endif %}
      <p>{{ suggestion.reason }}</p>
      {% if suggestion.evidence %}<p>근거 인용: “{{ suggestion.evidence }}”</p>{% endif %}
      <small>AI 제안은 자동 적용되지 않습니다. 최종 RASS는 간호사가 확정합니다.</small>
    </div>{% endif %}
  </section>

  <section class="card step-card">
    <h2 class="step-title"><span class="step-num">2</span> 표준 절차 관찰 (간호사 판정)</h2>
    <p class="muted">단계별로 한 항목씩 선택하세요. 반응이 없으면 다음 자극 단계가 열립니다.</p>
    <input type="hidden" name="observation_index" id="observation-index" value="{{ form_data.get('observation_index','') }}">

    <h3>① 무자극 관찰</h3>
    {% for text, score in obs_options[:5] %}
    <label class="obs-option"><input type="radio" name="stage_none" value="{{ loop.index0 }}" data-score="{{ score }}"><span class="obs-label">{{ text }}</span></label>
    {% endfor %}
    <label class="obs-option"><input type="radio" name="stage_none" value="next"><span class="obs-label">깨어 있지 않음 → 음성 자극으로</span></label>

    <div id="stage-voice" style="display:none">
      <h3>② 음성 자극</h3>
      {% for text, score in obs_options[5:8] %}
      <label class="obs-option"><input type="radio" name="stage_voice" value="{{ loop.index0 + 5 }}" data-score="{{ score }}"><span class="obs-label">{{ text }}</span></label>
      {% endfor %}
      <label class="obs-option"><input type="radio" name="stage_voice" value="next"><span class="obs-label">목소리에 반응 없음 → 신체 자극으로</span></label>
    </div>

    <div id="stage-body" style="display:none">
      <h3>③ 신체 자극</h3>
      {% for text, score in obs_options[8:] %}
      <label class="obs-option"><input type="radio" name="stage_body" value="{{ loop.index0 + 8 }}" data-score="{{ score }}"><span class="obs-label">{{ text }}</span></label>
      {% endfor %}
    </div>
  </section>

  <section class="card step-card">
    <h2 class="step-title"><span class="step-num">3</span> 최종 RASS 확정</h2>
    <p class="muted">2단계 선택에 따라 자동으로 채워집니다. 확인 후 필요하면 수정하세요.</p>
    <label>최종 RASS 점수
      <select name="rass_score" id="rass-select" required>
        <option value="">선택하세요</option>
        {% for n in range(4,-6,-1) %}<option value="{{ n }}" {% if form_data.get('rass_score')==n|string %}selected{% endif %}>{{ '%+d'|format(n) }}</option>{% endfor %}
      </select>
    </label>
    <div id="mismatch" class="notice" style="display:none;background:#fff4e5;color:#8a4b00">⚠️ AI 제안과 다릅니다. 관찰 서술과 선택을 다시 확인하세요.</div>
    <p class="notice">−4 또는 −5를 선택하면 CAM-ICU는 UNASSESSABLE로 저장됩니다.</p>
  </section>

  <section class="card step-card" id="cam-section">
    <h2 class="step-title"><span class="step-num">4</span> CAM-ICU 특징 평가</h2>
    {% set labels={1:'F1: 급성 변화 또는 변동 경과',2:'F2: 주의력 장애',4:'F4: 비체계적 사고'} %}
    {% for i in [1,2,4] %}
    <fieldset><legend>{{ labels[i] }}</legend>
      <label class="radio"><input type="radio" name="feature{{ i }}" value="yes" {% if form_data.get('feature'~i)=='yes' %}checked{% endif %}> 있음</label>
      <label class="radio"><input type="radio" name="feature{{ i }}" value="no"  {% if form_data.get('feature'~i)=='no'  %}checked{% endif %}> 없음</label>
    </fieldset>{% endfor %}
    <p class="formula">양성 공식: F1 AND F2 AND (F3 OR F4) — F3는 RASS가 0이 아니면 자동 양성, 판정은 서버에서만 계산됩니다</p>
    <div id="unassessable-notice" class="notice" style="display:none">RASS −4·−5 선택 시 UNASSESSABLE로 저장됩니다.</div>
  </section>

  <section class="card step-card">
    <h2 class="step-title"><span class="step-num">5</span> 메모 (선택)</h2>
    <textarea name="memo" rows="2" placeholder="진정제 용량 변경, 특이 관찰 사항 등 자유 입력..." style="width:100%;padding:10px;border:1px solid #bdcbd9;border-radius:9px;font:inherit;resize:vertical">{{ form_data.get('memo','') }}</textarea>
  </section>

  <section class="card step-card">
    <h2 class="step-title"><span class="step-num">6</span> 최종 평가 저장</h2>
    <button class="submit" name="action" value="save" type="submit">평가 결과 저장</button>
  </section>
</form>

<script>
(function(){
  var sel=document.getElementById('rass-select');
  var cam=document.getElementById('cam-section');
  var inputs=cam.querySelectorAll('input');
  var notice=document.getElementById('unassessable-notice');
  var aiBox=document.getElementById('ai-box');
  var mismatch=document.getElementById('mismatch');
  function compare(){
    var ai=aiBox?aiBox.dataset.score:'';
    mismatch.style.display=(ai!==''&&sel.value!==''&&ai!==sel.value)?'block':'none';
  }
  function gate(){compare();var locked=sel.value!==''&&parseInt(sel.value,10)<=-4;
    cam.classList.toggle('disabled',locked);inputs.forEach(function(x){x.disabled=locked;});
    notice.style.display=locked?'block':'none';}
  sel.addEventListener('change',gate); gate();

  var hidden=document.getElementById('observation-index');
  var voice=document.getElementById('stage-voice');
  var body=document.getElementById('stage-body');
  function picked(name){var el=document.querySelector('input[name='+name+']:checked');return el?el.value:'';}
  function clearStage(name){document.querySelectorAll('input[name='+name+']').forEach(function(x){x.checked=false;});}
  function sync(){
    var first=picked('stage_none');
    if(first!=='next'){clearStage('stage_voice');}
    var second=picked('stage_voice');
    if(second!=='next'){clearStage('stage_body');}
    voice.style.display=first==='next'?'block':'none';
    body.style.display=second==='next'?'block':'none';
    var result=first;
    if(first==='next'){result=second==='next'?picked('stage_body'):second;}
    var changed=hidden.value!==result;
    hidden.value=result;
    var chosen=document.querySelector('input[name^=stage_][value="'+result+'"]');
    if(changed&&chosen&&chosen.dataset.score!==undefined){sel.value=chosen.dataset.score;gate();}
    document.querySelectorAll('.obs-option').forEach(function(el){
      el.classList.toggle('obs-selected',el.querySelector('input').checked);});
  }
  function check(name,value){var el=document.querySelector('input[name='+name+'][value="'+value+'"]');if(el)el.checked=true;}
  var saved=parseInt(hidden.value,10);
  if(saved>=0&&saved<=4){check('stage_none',saved);}
  if(saved>=5){check('stage_none','next');}
  if(saved>=5&&saved<=7){check('stage_voice',saved);}
  if(saved>=8){check('stage_voice','next');check('stage_body',saved);}
  document.querySelectorAll('input[name^=stage_]').forEach(function(r){r.addEventListener('change',sync);});
  sync();
}());
</script>
{% endblock %}
''')


In [ ]:
write('templates/base.html', r'''<!doctype html><html lang="ko"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{% block title %}KY CAM-ICU/RASS{% endblock %}</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
<link rel="manifest" href="{{ url_for('static', filename='manifest.webmanifest') }}">
<meta name="theme-color" content="#0f6e86">
<link rel="apple-touch-icon" href="{{ url_for('static', filename='icons/icon-192.png') }}">
</head><body>
<header>
  <a class="brand" href="{% if current_user and current_user.role=='ADMIN' %}{{ url_for('admin') }}{% else %}{{ url_for('dashboard') }}{% endif %}">KY CAM-ICU · RASS</a>
  {% if current_user %}<nav>
    {% if current_user.role=='ADMIN' %}
      <a href="{{ url_for('admin') }}">간호사 관리</a>
      <a href="{{ url_for('admin_patients') }}">환자 관리</a>
      <a href="{{ url_for('stats') }}">통계</a>
    {% else %}
      <a href="{{ url_for('handover') }}">📝 인계</a>
      <a href="{{ url_for('stats') }}">📊 통계</a>
      <a href="{{ url_for('mypage') }}">마이페이지</a>
    {% endif %}
    <a href="{{ url_for('logout') }}">로그아웃</a>
  </nav>{% endif %}
</header>
{% if current_user and current_user.role=='NURSE' %}
<div class="info-bar">
  <span>🏥 KY병원</span>
  <span>사번: {{ current_user.employee_no }}</span>
  <span>{{ current_user.ward_name }}</span>
  <span>{{ current_user.name }} 간호사</span>
</div>
{% endif %}
<main>
  {% for cat,msg in get_flashed_messages(with_categories=true) %}
  <div class="alert {{ cat }}">{{ msg }}</div>{% endfor %}
  {% block content %}{% endblock %}
</main>
<footer>교육·시연용 시스템 · 실제 임상 판단을 대체하지 않습니다.</footer>
<script>
if ('serviceWorker' in navigator) {
  navigator.serviceWorker.register('/sw.js');
}
</script>
</body></html>
''')


In [ ]:
write('templates/dashboard.html', r'''{% extends 'base.html' %}
{% block content %}
<div class="page-head">
  <div>
    <h1>담당 환자</h1>
    <p class="muted">{{ patients|length }}명</p>
  </div>
</div>

<!-- 검색 + 정렬 -->
<div class="toolbar">
  <input id="search" type="text" placeholder="🔍 이름 또는 병실 검색..." class="search-input">
  <div class="sort-btns">
    {% for key, (label, _) in sort_options.items() %}
    <a class="sort-btn{% if sort==key %} active{% endif %}" href="{{ url_for('dashboard', sort=key) }}">{{ label }}</a>
    {% endfor %}
  </div>
</div>

<!-- 환자 테이블 (가로 정렬) -->
<div class="table-wrap card">
  <table id="patient-table">
    <thead>
      <tr>
        <th>병실</th>
        <th>이름</th>
        <th>나이·성별</th>
        <th>최근 결과</th>
        <th>검사 시간</th>
        <th>다음 검사까지</th>
        <th>상태</th>
        <th></th>
      </tr>
    </thead>
    <tbody>
      {% for p in patients %}
      <tr data-name="{{ p.name }}" data-room="{{ p.room_number }}"
          class="{% if p.status=='퇴원' %}row-discharged{% elif p.status=='외출' %}row-out{% endif %}">
        <td class="room">{{ p.room_number }}</td>
        <td><a href="{{ url_for('patient_detail', patient_id=p.id) }}">{{ p.name }}</a></td>
        <td>{{ p.age }}세 · {{ p.sex }}</td>
        <td>
          {% if p.last_res %}
          <span class="badge {{ p.last_res|lower }}">{{ p.last_res }}</span>
          {% else %}
          <span class="badge neutral">미평가</span>
          {% endif %}
        </td>
        <td class="time-cell">
          {% if p.last_at %}
          <span class="last-time" data-utc="{{ p.last_at }}">{{ p.last_at[:16] }}</span>
          {% else %}<span class="muted">—</span>{% endif %}
        </td>
        <td>
          {% if p.last_res == 'UNASSESSABLE' %}
          <span class="countdown special">진정 변화 시 재평가</span>
          {% elif p.last_at and p.last_res %}
          <span class="countdown" data-assessed="{{ p.last_at }}" data-result="{{ p.last_res }}">계산 중…</span>
          {% else %}
          <span class="countdown overdue">즉시 평가 필요</span>
          {% endif %}
        </td>
        <td>
          <span class="status-tag
            {%- if p.status=='입원중' %} s-in
            {%- elif p.status=='외출' %} s-out
            {%- else %} s-dis{%- endif %}">{{ p.status }}</span>
        </td>
        <td>
          {% if p.status != '퇴원' %}
          <a class="button btn-sm" href="{{ url_for('assessment', patient_id=p.id) }}">평가</a>
          {% endif %}
        </td>
      </tr>
      {% endfor %}
    </tbody>
  </table>
  <p id="no-result" class="empty" style="display:none">검색 결과가 없습니다.</p>
</div>

<script>
// ── 검색 ──────────────────────────────────────────────
const searchInput = document.getElementById('search');
searchInput.addEventListener('input', function () {
  const q = this.value.toLowerCase();
  let visible = 0;
  document.querySelectorAll('#patient-table tbody tr').forEach(function (row) {
    const match = row.dataset.name.toLowerCase().includes(q) ||
                  row.dataset.room.toLowerCase().includes(q);
    row.style.display = match ? '' : 'none';
    if (match) visible++;
  });
  document.getElementById('no-result').style.display = visible === 0 ? 'block' : 'none';
});

// ── 검사 시간 현지 시간으로 표시 ──────────────────────
document.querySelectorAll('.last-time[data-utc]').forEach(function (el) {
  const d = new Date(el.dataset.utc.replace(' ', 'T') + 'Z');
  el.textContent = d.toLocaleString('ko-KR', {month:'2-digit',day:'2-digit',
    hour:'2-digit',minute:'2-digit'});
});

// ── 카운트다운 ────────────────────────────────────────
function intervalMs(result) {
  if (result === 'POSITIVE')  return 4 * 3600000;   // 4시간
  if (result === 'NEGATIVE')  return 8 * 3600000;   // 8시간
  return null;
}
function updateCountdowns() {
  document.querySelectorAll('.countdown[data-assessed]').forEach(function (el) {
    const assessed = new Date(el.dataset.assessed.replace(' ', 'T') + 'Z');
    const ms = intervalMs(el.dataset.result);
    if (!ms) return;
    const remaining = assessed.getTime() + ms - Date.now();
    if (remaining <= 0) {
      el.textContent = '재평가 필요!';
      el.className = 'countdown overdue';
    } else {
      const h = Math.floor(remaining / 3600000);
      const m = Math.floor((remaining % 3600000) / 60000);
      el.textContent = h + '시간 ' + m + '분 남음';
      el.className = 'countdown ' + (h >= 2 ? 'ok' : h >= 1 ? 'warn' : 'urgent');
    }
  });
}
setInterval(updateCountdowns, 30000);
updateCountdowns();
</script>
{% endblock %}
''')


In [ ]:
write('templates/error.html', r'''{% extends 'base.html' %}{% block content %}<section class="result-card"><h1>{{ code }}</h1><p>{{ message }}</p><a class="button" href="{{ url_for('dashboard') }}">환자 목록으로</a></section>{% endblock %}
''')


In [ ]:
write('templates/handover.html', r'''{% extends 'base.html' %}
{% block content %}
<h1>근무 인계 요약</h1>
<p class="muted">담당 환자 {{ items|length }}명 · 확인이 먼저 필요한 환자부터 표시합니다.</p>
<section class="card">
  <div class="table-wrap"><table>
    <thead><tr><th>병실</th><th>이름</th><th>최근 결과</th><th>RASS</th><th>최근 평가</th><th>재평가 마감</th><th>확인 사항</th><th>메모</th></tr></thead>
    <tbody>
    {% for item in items %}
      <tr>
        <td>{{ item.patient.room_number }}</td>
        <td><a href="{{ url_for('patient_detail', patient_id=item.patient.id) }}">{{ item.patient.name }}</a></td>
        {% if item.latest %}
          <td><span class="badge {{ item.latest.cam_result|lower }}">{{ item.latest.cam_result }}</span></td>
          <td>{{ '%+d'|format(item.latest.rass_score) }}</td>
        {% else %}
          <td><span class="badge neutral">미평가</span></td><td>—</td>
        {% endif %}
        <td>{{ item.assessed_text }}</td>
        <td>{{ item.due_text }}</td>
        <td>{{ item.flags|join(' · ') if item.flags else '—' }}</td>
        <td>{{ item.latest.memo if item.latest and item.latest.memo else '—' }}</td>
      </tr>
    {% else %}
      <tr><td colspan="8">담당 환자가 없습니다.</td></tr>
    {% endfor %}
    </tbody>
  </table></div>
</section>
{% endblock %}
''')


In [ ]:
write('templates/login.html', r'''{% extends 'base.html' %}
{% block title %}KY 로그인{% endblock %}
{% block content %}
<section class="login-card">
  <h1>KY 로그인</h1>
  <p class="muted">간호사 및 관리자 계정으로 로그인하세요.</p>
  <form method="post">
    <label>아이디<input name="username" required autocomplete="username"></label>
    <label>비밀번호<input type="password" name="password" required autocomplete="current-password"></label>
    <button>로그인</button>
  </form>
  <div class="demo">
    <b>시연 계정</b><br>
    간호사: nurse01 / 1234 &nbsp;·&nbsp; nurse02 / 1234<br>
    관리자: admin / admin1234
  </div>
</section>
{% endblock %}
''')


In [ ]:
write('templates/mypage.html', r'''{% extends 'base.html' %}
{% block title %}마이페이지{% endblock %}
{% block content %}
<h1>마이페이지</h1>

<section class="profile card">
  <div class="avatar">{{ current_user.name[0] }}</div>
  <div>
    <h2>{{ current_user.name }}</h2>
    <p>{{ current_user.employee_no }} · {{ current_user.ward_name }}</p>
    <span class="badge neutral">간호사</span>
  </div>
</section>

<div class="stats">
  <div><b>{{ stats.total or 0 }}</b><span>전체 평가</span></div>
  <div><b>{{ stats.positive or 0 }}</b><span>양성</span></div>
  <div><b>{{ stats.negative or 0 }}</b><span>음성</span></div>
  <div><b>{{ stats.unassessable or 0 }}</b><span>평가 불가</span></div>
</div>

<!-- 환자 목록 + 정렬 -->
<section class="card">
  <div class="sort-bar">
    <span class="sort-label">내 환자 목록 ({{ patients|length }}명)</span>
    <div class="sort-btns">
      {% for key, (label, _) in sort_options.items() %}
      <a class="sort-btn{% if sort == key %} active{% endif %}"
         href="{{ url_for('mypage', sort=key) }}">{{ label }}</a>
      {% endfor %}
    </div>
  </div>

  {% if patients %}
  <div class="table-wrap">
    <table>
      <thead>
        <tr>
          <th>병실</th><th>이름</th><th>나이·성별</th><th>최근 평가</th><th>결과</th><th>상태</th><th></th>
        </tr>
      </thead>
      <tbody>
        {% for p in patients %}
        <tr class="{% if p.status == '퇴원' %}row-discharged{% elif p.status == '외출' %}row-out{% endif %}">
          <td class="room">{{ p.room_number }}</td>
          <td><a href="{{ url_for('patient_detail', patient_id=p.id) }}">{{ p.name }}</a></td>
          <td>{{ p.age }}세 · {{ p.sex }}</td>
          <td>{% if p.last_at %}{{ p.last_at[:16] }}{% else %}<span class="muted">미평가</span>{% endif %}</td>
          <td>
            {% if p.last_res %}
            <span class="badge {{ p.last_res|lower }}">{{ p.last_res }}</span>
            {% else %}
            <span class="badge neutral">—</span>
            {% endif %}
          </td>
          <td>
            <form method="post" action="{{ url_for('update_patient_status', patient_id=p.id) }}" class="status-form">
              <select name="status" onchange="this.form.submit()" class="status-sel
                {%- if p.status == '입원중' %} status-in
                {%- elif p.status == '외출' %} status-out
                {%- else %} status-discharged{%- endif %}">
                <option{% if p.status == '입원중' %} selected{% endif %}>입원중</option>
                <option{% if p.status == '외출' %} selected{% endif %}>외출</option>
                <option{% if p.status == '퇴원' %} selected{% endif %}>퇴원</option>
              </select>
            </form>
          </td>
          <td>
            {% if p.status != '퇴원' %}
            <a class="button btn-sm" href="{{ url_for('assessment', patient_id=p.id) }}">평가</a>
            {% endif %}
          </td>
        </tr>
        {% endfor %}
      </tbody>
    </table>
  </div>
  {% else %}
  <p class="empty">담당 환자가 없습니다.</p>
  {% endif %}
</section>
{% endblock %}
''')


In [ ]:
write('templates/nurse_register.html', r'''{% extends 'base.html' %}
{% block title %}간호사 추가{% endblock %}
{% block content %}
<a class="back" href="{{ url_for('admin') }}">← 관리자 페이지</a>
<section class="login-card" style="max-width:560px">
  <h1>간호사 계정 추가</h1>
  <p class="muted">회원가입 완료 후 바로 로그인 가능합니다.</p>
  <form method="post">
    <label>이름<input name="name" required placeholder="홍길동"></label>
    <label>아이디<input name="username" required placeholder="nurse03" autocomplete="off"></label>
    <label>사번<input name="employee_no" required placeholder="N-1003"></label>
    <label>소속 병동
      <select name="ward_id" required>
        <option value="">선택하세요</option>
        {% for w in wards %}
        <option value="{{ w.id }}">{{ w.name }}</option>
        {% endfor %}
      </select>
    </label>
    <label>비밀번호<input type="password" name="password" required minlength="4" autocomplete="new-password"></label>
    <label>비밀번호 확인<input type="password" name="confirm" required autocomplete="new-password"></label>
    <button>계정 생성</button>
  </form>
</section>
{% endblock %}
''')


In [ ]:
write('templates/patient_detail.html', r'''{% extends 'base.html' %}
{% block content %}
<div class="page-head">
  <div>
    <a class="back" href="{{ url_for('dashboard') }}">← 환자 목록</a>
    <h1>{{ patient.name }}</h1>
    <p>{{ patient.patient_code }} · {{ patient.room_number }} · {{ patient.age }}세 · {{ patient.sex }}</p>
  </div>
  <a class="button" href="{{ url_for('assessment', patient_id=patient.id) }}">새 평가</a>
</div>

<!-- 꺾은선 그래프 -->
{% if records %}
<section class="card" style="margin-bottom:18px">
  <h2>RASS 추이</h2>
  <div style="position:relative;height:220px">
    <canvas id="rass-chart"></canvas>
  </div>
  <div class="chart-legend">
    <span class="legend-dot" style="background:#d64545"></span>POSITIVE &nbsp;
    <span class="legend-dot" style="background:#23865f"></span>NEGATIVE &nbsp;
    <span class="legend-dot" style="background:#654d96"></span>UNASSESSABLE
  </div>
</section>
{% endif %}

<!-- 평가 기록 테이블 -->
<section class="card">
  <h2>평가 기록</h2>
  {% if records %}
  <div class="table-wrap">
    <table>
      <thead><tr><th>일시</th><th>간호사</th><th>RASS</th><th>CAM-ICU</th><th></th></tr></thead>
      <tbody>
        {% for r in records %}
        <tr>
          <td>{{ r.assessed_at[:16] }}</td>
          <td>{{ r.nurse_name }}</td>
          <td>{{ '%+d'|format(r.rass_score) }}</td>
          <td><span class="badge {{ r.cam_result|lower }}">{{ r.cam_result }}</span></td>
          <td><a href="{{ url_for('result', assessment_id=r.id) }}">상세</a></td>
        </tr>
        {% endfor %}
      </tbody>
    </table>
  </div>
  {% else %}
  <p class="empty">아직 평가 기록이 없습니다.</p>
  {% endif %}
</section>

{% if records %}
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
(function () {
  var raw = {{ chart_data | safe }};
  // 시간 라벨 현지 시간으로 변환
  var labels = raw.labels.map(function (l) {
    var d = new Date(l.replace(' ', 'T') + 'Z');
    return d.toLocaleString('ko-KR', {month:'2-digit',day:'2-digit',
      hour:'2-digit',minute:'2-digit'});
  });
  new Chart(document.getElementById('rass-chart'), {
    type: 'line',
    data: {
      labels: labels,
      datasets: [{
        label: 'RASS',
        data: raw.rass,
        borderColor: '#3979d8',
        backgroundColor: 'rgba(57,121,216,0.08)',
        borderWidth: 2,
        tension: 0.3,
        fill: true,
        pointBackgroundColor: raw.colors,
        pointBorderColor: raw.colors,
        pointRadius: 7,
        pointHoverRadius: 9,
      }]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        tooltip: {
          callbacks: {
            label: function (ctx) {
              return 'RASS ' + (ctx.raw >= 0 ? '+' : '') + ctx.raw +
                     '  (' + raw.results[ctx.dataIndex] + ')';
            }
          }
        }
      },
      scales: {
        y: {
          min: -5, max: 4,
          ticks: { stepSize: 1,
            callback: function (v) { return (v >= 0 ? '+' : '') + v; }
          },
          grid: { color: 'rgba(0,0,0,0.06)' }
        },
        x: { grid: { display: false } }
      }
    }
  });
}());
</script>
{% endif %}
{% endblock %}
''')


In [ ]:
write('templates/result.html', r'''{% extends 'base.html' %}{% block content %}
<a class="back" href="{{ url_for('patient_detail', patient_id=row.patient_id) }}">← 평가 기록</a>
<section class="result-card">
  <p>{{ row.patient_name }} · {{ row.room_number }}</p>
  <h1 class="result {{ row.cam_result|lower }}">{{ row.cam_result }}</h1>
  {% if row.cam_result == 'UNASSESSABLE' %}
  <p class="unassessable-msg">RASS {{ row.rass_score }}로 인해 CAM-ICU 평가가 불가능합니다.</p>
  {% else %}
  <p>RASS {{ '%+d'|format(row.rass_score) }} · 서버 공식으로 계산됨</p>
  <div class="feature-row">
    <span>F1 {{ '있음' if row.feature1 else '없음' }}</span>
    <span>F2 {{ '있음' if row.feature2 else '없음' }}</span>
    <span>F3 {{ '있음' if row.feature3 else '없음' }}</span>
    <span>F4 {{ '있음' if row.feature4 else '없음' }}</span>
  </div>
  {% endif %}
  <dl>
    <dt>평가자</dt><dd>{{ row.nurse_name }}</dd>
    <dt>평가 시각</dt><dd>{{ row.assessed_at }}</dd>
    <dt>표준 절차 관찰</dt><dd>{{ row.observation }}</dd>
    <dt>관찰 서술</dt><dd>{{ row.narrative or '—' }}</dd>
    <dt>AI 해석</dt>
    <dd>{% if row.ai_status == 'SCORED' %}제안 RASS {{ '%+d'|format(row.ai_rass) }}
          · {{ '간호사 확정과 일치' if row.ai_rass == row.rass_score else '간호사가 다르게 확정' }}
        {% elif row.ai_status == 'NOT_RUN' %}사용 안 함
        {% else %}{{ {'ABSTAIN':'판단 보류','UNAVAILABLE':'사용 불가','ERROR':'오류'}.get(row.ai_status, row.ai_status) }}{% endif %}
        {% if row.ai_reason %}<br><small>{{ row.ai_reason }}</small>{% endif %}
        {% if row.ai_evidence %}<br><small>근거 인용: “{{ row.ai_evidence }}”</small>{% endif %}</dd>
  </dl>
</section>
{% endblock %}
''')


In [ ]:
write('templates/stats.html', r'''{% extends 'base.html' %}
{% block title %}통계{% endblock %}
{% block content %}
<h1>{{ scope }} 통계</h1>
{% if not is_admin %}<p class="muted">소속 병동 집계만 표시합니다. 환자 이름은 담당 환자만 보입니다.</p>{% endif %}

<!-- 요약 카드 -->
<div class="stats">
  <div><b>{{ total_p | int }}</b><span>총 환자</span></div>
  <div><b>{{ total_a | int }}</b><span>총 평가 건수</span></div>
  <div><b>{{ pos_rate }}%</b><span>섬망 양성률</span></div>
  <div><b>{{ recent_7 }}</b><span>최근 7일 평가</span></div>
</div>

<!-- 차트 2×2 -->
<div class="chart-grid">
  <section class="card"><h2>CAM-ICU 결과 분포</h2><div class="chart-wrap"><canvas id="c-cam"></canvas></div></section>
  <section class="card"><h2>RASS 점수 분포</h2><div class="chart-wrap"><canvas id="c-rass"></canvas></div></section>
  <section class="card chart-wide"><h2>주별 섬망 발생 추이 (최근 12주)</h2><div class="chart-wrap-wide"><canvas id="c-weekly"></canvas></div></section>
  <section class="card"><h2>병동별 현황</h2><div class="chart-wrap"><canvas id="c-ward"></canvas></div></section>
</div>

<!-- 병동별 테이블 -->
<section class="card" style="margin-top:18px">
  <h2>병동별 상세</h2>
  <table>
    <thead><tr><th>병동</th><th>환자 수</th><th>평가 건수</th><th>POSITIVE</th><th>양성률</th></tr></thead>
    <tbody>
      {% for w in ward_stats %}
      <tr>
        <td>{{ w.name }}</td>
        <td>{{ w.patients }}</td>
        <td>{{ w.assessments }}</td>
        <td><span class="badge positive">{{ w.positive or 0 }}</span></td>
        <td>{{ '%.1f'|format((w.positive or 0)/w.assessments*100) if w.assessments else '—' }}%</td>
      </tr>
      {% endfor %}
    </tbody>
  </table>
</section>

<!-- POSITIVE 상위 환자 -->
{% if top_pos %}
<section class="card" style="margin-top:18px">
  <h2>POSITIVE 상위 환자 (Top 10){% if not is_admin %} · 내 담당{% endif %}</h2>
  <table>
    <thead><tr><th>이름</th><th>병실</th><th>POSITIVE 횟수</th></tr></thead>
    <tbody>
      {% for p in top_pos %}
      <tr><td>{{ p.name }}</td><td>{{ p.room_number }}</td><td><span class="badge positive">{{ p.cnt }}회</span></td></tr>
      {% endfor %}
    </tbody>
  </table>
</section>
{% endif %}

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
var D = {{ chart | safe }};
var cam = D.cam;
// CAM 도넛
new Chart(document.getElementById('c-cam'),{type:'doughnut',data:{
  labels:['POSITIVE','NEGATIVE','UNASSESSABLE'],
  datasets:[{data:[cam.POSITIVE||0,cam.NEGATIVE||0,cam.UNASSESSABLE||0],
    backgroundColor:['#d64545','#23865f','#654d96'],borderWidth:2}]
},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{position:'bottom'}}}});

// RASS 막대
new Chart(document.getElementById('c-rass'),{type:'bar',data:{
  labels:D.rass.labels.map(function(v){return(v>=0?'+':'')+v;}),
  datasets:[{label:'건수',data:D.rass.data,backgroundColor:'#3979d8',borderRadius:4}]
},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
  scales:{y:{beginAtZero:true},x:{grid:{display:false}}}}});

// 주별 꺾은선
new Chart(document.getElementById('c-weekly'),{type:'line',data:{
  labels:D.weekly.labels,
  datasets:[
    {label:'전체 평가',data:D.weekly.total,borderColor:'#3979d8',backgroundColor:'rgba(57,121,216,0.1)',fill:true,tension:0.3},
    {label:'POSITIVE',data:D.weekly.positive,borderColor:'#d64545',backgroundColor:'rgba(214,69,69,0.08)',fill:true,tension:0.3}
  ]
},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{position:'bottom'}},
  scales:{y:{beginAtZero:true},x:{grid:{display:false},ticks:{maxRotation:45}}}}});

// 병동 막대
new Chart(document.getElementById('c-ward'),{type:'bar',data:{
  labels:D.wards.labels,
  datasets:[
    {label:'환자',data:D.wards.patients,backgroundColor:'#3979d8',borderRadius:4},
    {label:'평가',data:D.wards.assessments,backgroundColor:'#23865f',borderRadius:4}
  ]
},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{position:'bottom'}},
  scales:{y:{beginAtZero:true},x:{grid:{display:false}}}}});
</script>
{% endblock %}
''')


In [ ]:
write('tests/ai_cases.csv', r'''narrative,expected
목소리에 눈 뜨고 10초 이상 눈맞춤 유지함,-1
이름 부르니 눈 떴는데 5초쯤 보다가 다시 감음,-2
불러도 눈맞춤은 없고 몸만 조금 움직임,-3
불러도 반응 없고 어깨를 흔들자 눈을 뜸,-4
목소리와 흉골 자극 모두 반응 없음,-5
깨어 있고 차분하게 질문에 답함,0
깨어 있으나 불안해하며 계속 두리번거림. 공격적이지 않음,1
목적 없이 계속 몸을 뒤척이고 인공호흡기와 맞지 않음,2
기관내관을 잡아당기려 하고 간호사에게 공격적임,3
간호사를 때리려 하는 등 폭력적임,4
진정제 감량 중. 오전보다 조금 나아 보임,
눈을 뜸,
''')


In [ ]:
write('tests/test_ai.py', r'''import os
import sys
import unittest
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import ai_rass
from ai_rass import parse_response, suggest_rass_ai

NARRATIVE = "이름 부르니 눈 떴는데 5초쯤 보다가 다시 감음"


class AiRassTest(unittest.TestCase):
    def setUp(self):
        self.saved = {k: os.environ.pop(k, None) for k in
                      ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "AI_PROVIDER")}

    def tearDown(self):
        for k, v in self.saved.items():
            if v is not None:
                os.environ[k] = v

    def test_scored_with_quoted_evidence(self):
        r = parse_response('{"score": -2, "criterion": "가벼운 진정", "evidence": "5초쯤 보다가 다시 감음", "missing": ""}', NARRATIVE)
        self.assertEqual((r["status"], r["score"]), ("SCORED", -2))

    def test_hallucinated_evidence_blocked(self):
        r = parse_response('{"score": -2, "criterion": "x", "evidence": "눈맞춤 3초 유지", "missing": ""}', NARRATIVE)
        self.assertEqual((r["status"], r["score"]), ("ABSTAIN", None))

    def test_out_of_range_and_bad_json(self):
        self.assertEqual(parse_response('{"score": 7, "evidence": "눈 떴는데"}', NARRATIVE)["status"], "ERROR")
        self.assertEqual(parse_response('{"score": "-2", "evidence": "눈 떴는데"}', NARRATIVE)["status"], "ERROR")
        self.assertEqual(parse_response("잘 모르겠습니다", NARRATIVE)["status"], "ERROR")

    def test_null_score_abstains(self):
        r = parse_response('{"score": null, "missing": "자극 단계 정보 없음"}', NARRATIVE)
        self.assertEqual(r["status"], "ABSTAIN")
        self.assertIn("자극 단계", r["reason"])

    def test_no_key_unavailable(self):
        self.assertEqual(suggest_rass_ai(NARRATIVE)["status"], "UNAVAILABLE")

    def test_empty_narrative(self):
        self.assertEqual(suggest_rass_ai("  ")["status"], "ABSTAIN")

    def test_with_fake_llm(self):
        os.environ["GEMINI_API_KEY"] = "test"
        fake = lambda provider, prompt: '```json\\n{"score": -2, "criterion": "가벼운 진정", "evidence": "눈 떴는데 5초쯤 보다가", "missing": ""}\\n```'
        r = suggest_rass_ai(NARRATIVE, caller=fake)
        self.assertEqual((r["status"], r["score"]), ("SCORED", -2))

    def test_prompt_contains_criteria_and_text(self):
        prompt = ai_rass.build_prompt(NARRATIVE)
        self.assertIn(NARRATIVE, prompt)
        self.assertIn("-5:", prompt)


if __name__ == "__main__":
    unittest.main(verbosity=2)
''')


In [ ]:
write('tests/test_logic.py', r'''import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))

import unittest
from cam_icu_logic import calculate_cam_icu, suggest_rass_by_index, OBSERVATION_OPTIONS


class CamIcuLogicTest(unittest.TestCase):
    # ── calculate_cam_icu ──
    def test_positive_rass_zero_needs_f4(self):
        self.assertEqual(calculate_cam_icu(0, True, True, True), "POSITIVE")
        self.assertEqual(calculate_cam_icu(0, True, True, False), "NEGATIVE")

    def test_positive_rass_nonzero_by_f3(self):
        self.assertEqual(calculate_cam_icu(-1, True, True, False), "POSITIVE")
        self.assertEqual(calculate_cam_icu(1, True, True, False), "POSITIVE")

    def test_negative(self):
        self.assertEqual(calculate_cam_icu(-1, True, False, True), "NEGATIVE")
        self.assertEqual(calculate_cam_icu(-1, False, True, True), "NEGATIVE")

    def test_gate(self):
        self.assertEqual(calculate_cam_icu(-4), "UNASSESSABLE")
        self.assertEqual(calculate_cam_icu(-5), "UNASSESSABLE")

    def test_rass_minus3_uses_features(self):
        self.assertEqual(calculate_cam_icu(-3, True, True, False), "POSITIVE")
        self.assertEqual(calculate_cam_icu(-3, False, True, True), "NEGATIVE")

    def test_invalid_rass(self):
        with self.assertRaises(ValueError):
            calculate_cam_icu(5, True, True, True)

    def test_missing_features(self):
        with self.assertRaises(ValueError):
            calculate_cam_icu(0, True, True, None)

    # ── suggest_rass_by_index ──
    def test_suggest_by_index_scores(self):
        """OBSERVATION_OPTIONS 인덱스와 RASS 점수 매핑 검증"""
        expected = [4, 3, 2, 1, 0, -1, -2, -3, -4, -5]
        for idx, exp_score in enumerate(expected):
            result = suggest_rass_by_index(idx)
            self.assertEqual(result["score"], exp_score, f"index {idx}: expected {exp_score}")
            self.assertEqual(result["status"], "SCORED")

    def test_suggest_invalid_index(self):
        with self.assertRaises(ValueError):
            suggest_rass_by_index(-1)
        with self.assertRaises(ValueError):
            suggest_rass_by_index(10)

    def test_observation_options_count(self):
        self.assertEqual(len(OBSERVATION_OPTIONS), 10)


if __name__ == "__main__":
    unittest.main(verbosity=2)
''')


In [ ]:
from PIL import Image, ImageDraw

(ROOT / 'static/icons').mkdir(parents=True, exist_ok=True)
for size in (192, 512):
    image = Image.new('RGB', (size, size), '#0f6e86')
    draw = ImageDraw.Draw(image)
    margin = size // 5
    draw.rounded_rectangle((margin, margin, size - margin, size - margin), radius=size // 10, fill='white')
    draw.rectangle((size * 0.44, size * 0.3, size * 0.56, size * 0.7), fill='#0f6e86')
    draw.rectangle((size * 0.3, size * 0.44, size * 0.7, size * 0.56), fill='#0f6e86')
    image.save(ROOT / f'static/icons/icon-{size}.png')
print('아이콘 생성 완료')


In [ ]:
import subprocess, sys

for test in ('tests/test_logic.py', 'tests/test_ai.py'):
    result = subprocess.run([sys.executable, test], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'{test} 실패 — 위 메시지를 확인하세요.')
print('테스트 통과')


In [ ]:
%run /content/cam_icu_rass_v11/colab_start.py


## (선택) AI 정확도 평가
서버 셀을 멈춘 뒤 실행하세요. 키가 Secrets에 있어야 합니다.

In [ ]:
import os
from google.colab import userdata
for k in ('GEMINI_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass
!cd /content/cam_icu_rass_v11 && python eval_ai.py tests/ai_cases.csv
